# WaveSpike-PPG v3: learnable wavelet + spiking network for PPG AF detection (open-access data only)

**Pipeline:** PPG (50 Hz) → **learnable causal wavelet bank, one filter pair per level** → **learnable hard-threshold denoising** → *(optional: drop dead bands, decimate to 25 Hz)* → **±LIF encoder** → **recurrent LIF layer** → leaky readout.

### What changed from v2, and why
| Item | Change | Reason (from the v2 run on MIMIC PERform AF) |
|---|---|---|
| D1–D6 | New dataset `mimic3_ext_ppg` (MIMIC-III-Ext-PPG v1.1.0, PhysioNet credentialed): subject selection → segment download → 50 Hz cache. MIMIC PERform AF becomes an **external** test set after removing overlapping subjects. *Inactive: the project is now open-access only (see below).* | 35 subjects is too few to compare methods |
| P0.1 | Fully deterministic training (`CUBLAS_WORKSPACE_CONFIG`, `use_deterministic_algorithms`); fold 0 is trained twice and the two results must match exactly | The same config gave test AUROC 0.965 in one run and 0.707 in another |
| P0.2 | Early stopping on **validation loss** (class-weighted CE) instead of val macro-F1; both the best-val-loss and the last-epoch checkpoints are kept and reported | Val F1 on 7 subjects was noisy; val AUROC fell from epoch 0 |
| P0.3 | **Out-of-fold** pooled window-level AUROC/F1 + subject-level AUROC | Per-fold AUROC on 7 test subjects ranged 0.39–0.96 |
| P0.4 | Baselines on the same folds: interval-irregularity + logistic regression, non-spiking GRU twin, SNN without wavelet | Without baselines we can't tell whether the wavelet or the SNN contributes anything |
| P0.5 | Repeated CV (3 fold seeds × 2 training seeds), paired differences vs `full`, and a consistency verdict | ΔF1 = −0.024 ± 0.051, p = 0.31: no method difference was detectable |
| P1.6 | `ht_nonneg=True` (**new default**): b = softplus(raw) ≥ 0, so HT can only shrink coefficients | The learned b went negative, giving a gain of 1.09–1.42 at x → 0 (amplification, not denoising) |
| P1.7 | `drop_bands`, `snn_decim` | d1 and d2 encoder rates were 0.001 (dead bands) |
| P1.8 | Augmentation, spike dropout, `hidden=32` (all off by default, ablated as variants) | Train loss fell 0.76 → 0.23 while val AUROC fell (overfitting / subject memorisation) |
| P1.9 | Label-noise check on interval irregularity; optional cleaning of the **training** data only | Ext-PPG labels come from chart notes up to 15 min away from the waveform |

The full list of code changes and of dataset-layout assumptions is in the last cell (and in `CHANGES.md`).

### Open-access only
This project uses **only datasets that download without an account, application or data use agreement** (`OPEN_ACCESS_ONLY = True` in §2):
- **MIMIC PERform AF** (Zenodo): training and internal repeated CV.
- **PhysioNet/CinC Challenge 2015**: the D7 sanity task. It is now **on** by default and is not AF detection.

The MIMIC-III-Ext-PPG code (D1–D6) is kept, but it is blocked while `OPEN_ACCESS_ONLY = True`. There is also **no external test set**: no second open PPG dataset with AF labels has been confirmed, so D6 is inactive.

**Caveat:** MIMIC PERform AF has only 35 subjects. The v3 fixes (determinism, val-loss stopping, OOF metrics, baselines, repeated CV) make results trustworthy, but with 35 subjects the paired comparisons may well show "no consistent difference".

### How to run
1. **Runtime → Change runtime type → T4 GPU**, then **Run all**. Allow Google Drive access when asked. No secrets or accounts are needed.
2. Data downloads once into `MyDrive/wavespike_data/`. The ablation saves one JSON per run, so an interrupted session resumes where it stopped.

## §1 Datasets
| Dataset | Rhythms | Sensor / rate | Subjects | Labels | Access | Role in v3 |
|---|---|---|---|---|---|---|
| **MIMIC-III-Ext-PPG v1.1.0** (Moulaeifard, Charlton & Strodthoff, *Sci Data* 13, 668 (2026); doi:10.13026/r6k1-xt76) | 26 rhythm codes, incl. AF, SR, STACH, SBRAD | ICU finger PPG, 125 Hz, 30 s WFDB segments, no preprocessing applied | 6,189 | Rhythm entries from clinical charts, **up to 15 min from the waveform** (label noise expected) | PhysioNet **credentialed** (CITI course + DUA); downloaded with `wget` using Colab secrets | Not used (credentialed; blocked by `OPEN_ACCESS_ONLY`) |
| **MIMIC PERform AF** (Charlton et al.) | AF vs non-AF | ICU finger PPG, 125 Hz, 20 min each | 35 (19 AF / 16 non-AF) | Manual AF labels (Bashar et al., *IEEE Access* 2019) | Open, Zenodo 10.5281/zenodo.6807402, CC BY 4.0 | **Training / internal repeated 5-fold CV** (default `DATASET`) |
| PhysioNet/CinC Challenge 2015 | True vs false alarm (here only Tachycardia / Ventricular_Tachycardia) | ICU, 250 Hz, 5-min records; only records with a PLETH channel are used | ? | Annotated alarms | Open | Sanity task (D7, **on by default**). **Not AF detection**, reported separately |
| DeepBeat, UMass Simband, UConn MIMIC-III, Pulsewatch | see v2 | | | | Synapse | Not used (need a Synapse account and/or approval) |

Only open-access datasets are used (`OPEN_ACCESS_ONLY = True`). Selecting a non-open dataset stops the run with an explanation.

In [ ]:
# §1 Setup
import os, sys, subprocess
_CUBLAS_PRESET = "CUBLAS_WORKSPACE_CONFIG" in os.environ
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")      # P0.1: must be set before CUDA initialises
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "wfdb", "synapseclient"], check=False)
_TORCH_WAS_LOADED = "torch" in sys.modules

import math, time, json, random, copy, glob, zipfile, urllib.request, urllib.parse, re, shutil, hashlib, tempfile, importlib
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass, asdict, field
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from scipy.signal import resample_poly, find_peaks, butter, sosfiltfilt
from scipy.stats import wilcoxon
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import f1_score, roc_auc_score, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
import matplotlib.pyplot as plt

NOTEBOOK_T0 = time.time()
SMOKE = os.environ.get("WAVESPIKE_SMOKE") == "1"     # local CPU test of the notebook only; never set in Colab
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cuda.matmul.allow_tf32 = False      # fp32 everywhere, so the GPU and NumPy (Pi) models agree
torch.backends.cudnn.allow_tf32 = False

# P0.1 determinism
DETERMINISTIC = True
torch.backends.cudnn.deterministic = DETERMINISTIC
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(DETERMINISTIC)
if DEVICE.type == "cuda" and _TORCH_WAS_LOADED and not _CUBLAS_PRESET and torch.cuda.is_initialized():
    print("⚠ CUDA was initialised before CUBLAS_WORKSPACE_CONFIG was set. Use Runtime → Restart session, then Run all.")
print("torch", torch.__version__, "| device:", DEVICE, "| deterministic:", DETERMINISTIC,
      "| CUBLAS_WORKSPACE_CONFIG =", os.environ["CUBLAS_WORKSPACE_CONFIG"])

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

class StopForUser(RuntimeError):
    """Raised on purpose: the notebook needs a decision from you. Read the message, edit §2, run again."""

In [ ]:
# §2 Configuration
# OPEN ACCESS ONLY: this project uses only datasets that download without an account, application or data use agreement.
OPEN_ACCESS_ONLY = True
OPEN_DATASETS = {"mimic_perform_af": "MIMIC PERform AF (Zenodo, CC BY 4.0)",
                 "synthetic": "generated locally", "npz": "your own file",
                 "zenodo_wrist_af": "Long-term wrist PPG + ECG-annotated AF, Zenodo 10.5281/zenodo.23597128 (open, no account; CC BY-NC-SA 4.0: non-commercial, share-alike)"}
NON_OPEN = {"mimic3_ext_ppg": "PhysioNet credentialed access (CITI training + DUA)",
            "deepbeat": "Synapse account + access token", "uconn_simband": "Synapse + approval by the data owner",
            "uconn_mimic3": "Synapse + approval by the data owner", "pulsewatch": "Synapse, not yet public"}
DATASET   = os.environ.get("WAVESPIKE_DATASET", "mimic_perform_af")   # | "zenodo_wrist_af" | "synthetic" | "npz"   (non-open, blocked while OPEN_ACCESS_ONLY: see NON_OPEN)
USE_DRIVE = True                 # cache downloads + processed data + results on Google Drive
NPZ_PATH, NPZ_FS = "/content/drive/MyDrive/ppg_windows.npz", 50     # only for DATASET="npz" (keys X, y, subject)
SYNTH_KW = {}                    # extra arguments for load_synthetic (DATASET="synthetic" only)

# ---- MIMIC-III-Ext-PPG (D1–D6): credentialed, NOT used while OPEN_ACCESS_ONLY = True (kept for later) ----
N_SUBJECTS          = 600        # D3a: subjects picked before any segment is downloaded (about half with AF)
MAX_SEG_PER_SUBJECT = 40         # D3b: segments per subject, balanced AF/SR where possible
EXT_SELECT_SEED     = 0
EXT_MULTICLASS      = False      # D2: True -> {SR, AF, STACH, SBRAD}; default binary AF vs SR
# Fill these in from the D1 printout (first run stops and prints the README + metadata.csv columns).
# Never guessed: the notebook stops until all three are set and exist in metadata.csv.
EXT_SCHEMA = {"subject_col": None,   # MIMIC-III subject ID of each segment
              "label_col":   None,   # rhythm label of each segment
              "path_col":    None}   # location of each segment's WFDB record
EXT_PATH_TEMPLATE = "{path}"     # record path relative to the project root, extension stripped; fields {path}, {subject}
EXT_LABEL_CODES = {"SR": "SR", "AF": "AF", "STACH": "STACH", "SBRAD": "SBRAD"}   # value in label_col for each class;
                                 # checked for an EXACT match at runtime (stops otherwise)
EXT_DOWNLOAD_PARALLEL = 8        # parallel wget processes
EXTERNAL_DATASET = "mimic_perform_af" if DATASET == "mimic3_ext_ppg" else None    # D6 external test set
RUN_CINC2015 = True              # D7 open-access sanity task (not AF detection); CHANGED: on by default
LABEL_CLEAN_TRAIN = False        # P1.9: drop label/feature disagreements from TRAIN+VAL only; reported as its own variant

# ---- repeated CV (P0.5) and session budget ----
CV_SEEDS    = [0, 1, 2]          # fold seeds
TRAIN_SEEDS = [0, 1]             # training seeds
MAX_SESSION_H = float(os.environ.get("WAVESPIKE_MAX_H", 3.6))              # stop launching new ablation runs after this many hours (Colab T4 ~4 h); rerun to resume

@dataclass
class Config:
    fs: int = 50
    win_sec: int = 30            # overwritten per dataset (DeepBeat: 25 s)
    levels: int = 5              # bands ≈ 12.5-25, 6.3-12.5, 3.1-6.3, 1.6-3.1, 0.8-1.6 Hz + approx <0.8 Hz
    taps: int = 8
    wavelet_init: str = "db4"
    learn_wavelet: bool = True
    per_level: bool = True       # DeSpaWN: one learned filter per level
    use_wavelet: bool = True     # NEW. False = raw signal straight into the encoder (baseline P0.4c)
    use_ht: bool = True          # DeSpaWN: learnable hard-threshold denoising
    ht_alpha: float = 10.0       # DeSpaWN uses α = 10
    ht_nonneg: bool = True       # NEW, CHANGES v2 behaviour: b = softplus(raw) >= 0 (P1.6). False = unconstrained DeSpaWN
    ht_init: float = 0.02        # NEW. initial b when ht_nonneg (raw = inv_softplus(0.02)); the unconstrained HT starts at 0 as in v2
    drop_bands: list = field(default_factory=list)   # NEW (P1.7). band indices to drop: 0 = d1 ... levels-1 = d5, levels = approx
    snn_decim: int = 1           # NEW (P1.7). keep every n-th sample of the kept bands before the encoder
    arch: str = "snn"            # NEW. "snn" | "gru" (non-spiking twin, P0.4b)
    spike_dropout: float = 0.0   # NEW (P1.8). dropout on hidden spikes, training only
    augment: bool = False        # NEW (P1.8). amplitude, circular shift, noise, baseline-wander augmentation
    augment_plus: bool = False   # NEW (v3.1). time-warp (x0.85-1.15), slow amplitude modulation; training only; use together with augment
    tau_het: bool = False        # NEW (v3.1). hidden-layer leaks initialised log-uniformly over tau_min..tau_max seconds (default: all 0.95 per 50 Hz step = 0.39 s)
    tau_min: float = 0.2
    tau_max: float = 4.0
    hidden: int = 64
    n_classes: int = 2           # overwritten by the dataset
    surr_slope: float = 10.0
    epochs: int = 30
    patience: int = 8            # early stopping patience
    select_on: str = "val_loss"  # CHANGED (v2: val macro-F1). "val_loss" (class-weighted CE) | "val_f1"
    batch: int = 64
    max_windows_per_epoch: int = 4000   # class-balanced subsample per epoch (large datasets)
    lr: float = 2e-3
    lr_wavelet: float = 5e-4
    lambda_wav: float = 1.0
    lambda_rate: float = 1.0
    target_rate: float = 0.10
    n_folds: int = 5
    seed: int = 0

CFG = Config()
for _k, _v in json.loads(os.environ.get("WAVESPIKE_CFG", "{}")).items():     # v3.1: optional config overrides (unset = defaults)
    setattr(CFG, _k, _v); print("CFG override:", _k, "=", _v)
if SMOKE:                        # local CPU test: tiny everything (not used in Colab)
    CFG.epochs, CFG.patience, CFG.max_windows_per_epoch, CFG.batch, CFG.n_folds = 2, 1, 64, 32, 3
    CV_SEEDS, TRAIN_SEEDS = [0], [0]
    for _k, _v in json.loads(os.environ.get("WAVESPIKE_OVERRIDES", "{}")).items():
        globals()[_k] = _v
        print("SMOKE override:", _k, "=", _v)
    EXTERNAL_DATASET = "mimic_perform_af" if DATASET == "mimic3_ext_ppg" else None

if USE_DRIVE and IN_COLAB:
    from google.colab import drive; drive.mount("/content/drive")
    DATA_ROOT = "/content/drive/MyDrive/wavespike_data"
else:
    DATA_ROOT = os.environ.get("WAVESPIKE_DATA_ROOT", "/content/wavespike_data")
if OPEN_ACCESS_ONLY and DATASET not in OPEN_DATASETS:
    raise StopForUser(f"DATASET = {DATASET!r} is not open access ({NON_OPEN.get(DATASET, 'unknown access')}). This project "
                      f"uses open data only: pick one of {list(OPEN_DATASETS)}, or set OPEN_ACCESS_ONLY = False.")
for sub in ("raw", "processed", "results"): os.makedirs(f"{DATA_ROOT}/{sub}", exist_ok=True)
print("data root:", DATA_ROOT, "| dataset:", DATASET, "| external:", EXTERNAL_DATASET)

### Data loading
**v2 loaders** (unchanged): each one (1) downloads the raw files to Drive once, (2) cuts non-overlapping windows at the native rate and drops NaN or flat windows, (3) resamples to 50 Hz, (4) caches `X, y, subject` in `processed/<dataset>.npz`. In MIMIC PERform AF every window inherits its subject's label (AF or non-AF).

**MIMIC-III-Ext-PPG (D1–D4)** (credentialed: not run while `OPEN_ACCESS_ONLY = True`), in order:
1. **D1:** list the project, download `README*` and `metadata.csv` to `raw/mimic3_ext_ppg/`, and print the README, the columns and example values. The run **stops** until `EXT_SCHEMA` is filled in.
2. **D1/D2:** read the subject, label and path columns. Check that the label codes match **exactly** and that no value is compound, and keep only AF/SR rows (or SR/AF/STACH/SBRAD when `EXT_MULTICLASS=True`).
3. **D6 (part 1):** remove subjects that also appear in MIMIC PERform AF, **before** selection.
4. **D3:** pick `N_SUBJECTS` subjects (fixed seed, stratified by "has any AF segment", about half AF), then up to `MAX_SEG_PER_SUBJECT` segments each (round-robin over classes = balanced where possible). This is written to a **manifest CSV**, which later runs reuse.
5. Download 3 headers and check the layout (125 Hz, a PPG channel, 25–31 s long); stop on any mismatch. Then fetch every selected `.hea`, and after that the signal files those headers name. Files already on Drive are skipped.
6. **D4:** pick the PLETH/PPG channel by name. Drop segments with NaN, with zero variance, or with < 25 s of valid data (samples inside ≥ 1 s runs of identical values count as invalid). Resample 125 → 50 Hz with `resample_poly(up=2, down=5)` and cache to `processed/mimic3_ext_ppg_50hz_30s.npz`.

In [ ]:
def download_once(url, dst):
    if os.path.exists(dst) and os.path.getsize(dst) > 0:
        return dst
    print("downloading", url); tmp = dst + ".part"
    urllib.request.urlretrieve(url, tmp); os.replace(tmp, dst); return dst

def extract_once(zpath, out_dir):
    marker = os.path.join(out_dir, ".extracted")
    if not os.path.exists(marker):
        os.makedirs(out_dir, exist_ok=True)
        with zipfile.ZipFile(zpath) as z: z.extractall(out_dir)
        open(marker, "w").close()
    return out_dir

def synapse_sync_once(syn_id, out_dir):
    marker = os.path.join(out_dir, ".synced")
    if os.path.exists(marker): return out_dir
    import synapseclient, synapseutils
    try:
        from google.colab import userdata; token = userdata.get("SYNAPSE_AUTH_TOKEN")
    except Exception as e:
        raise RuntimeError("Add a Colab secret named SYNAPSE_AUTH_TOKEN (Synapse > Account Settings > Personal Access Tokens)") from e
    syn = synapseclient.Synapse(); syn.login(authToken=token, silent=True)
    os.makedirs(out_dir, exist_ok=True)
    synapseutils.syncFromSynapse(syn, syn_id, path=out_dir)
    open(marker, "w").close(); return out_dir

def inventory(root, max_files=40):
    # Print the first files and the arrays inside them, so a converter can be written for an unknown layout.
    files = sorted(p for p in glob.glob(f"{root}/**/*", recursive=True) if os.path.isfile(p) and not os.path.basename(p).startswith("."))
    print(f"{len(files)} files under {root}")
    for p in files[:max_files]:
        info = ""
        try:
            if p.endswith(".npz"):
                d = np.load(p, allow_pickle=False); info = {k: d[k].shape for k in d.files}
            elif p.endswith(".mat"):
                from scipy.io import whosmat; info = whosmat(p)
        except Exception as e:
            info = f"(unreadable: {e})"
        print(" ", os.path.relpath(p, root), os.path.getsize(p)//1024, "kB", info)

def windows_from_signal(sig, fs_in, fs_out, win_sec):
    # Non-overlapping windows at the native rate; drop NaN or flat windows; resample each kept window.
    n = int(round(win_sec*fs_in)); out = []
    g = math.gcd(int(round(fs_in)), fs_out); up, down = fs_out//g, int(round(fs_in))//g
    for i in range(len(sig)//n):
        w = np.asarray(sig[i*n:(i+1)*n], dtype=np.float64)
        if not np.all(np.isfinite(w)) or w.std() < 1e-6: continue
        out.append(resample_poly(w, up, down) if (up, down) != (1, 1) else w)
    return out

# ---------------- PhysioNet download helpers (D1, D3) ----------------
# Credentials come from Colab secrets and are only ever passed to wget as arguments. They are never printed:
# wget output is scrubbed before anything is shown.
FAKE_PHYSIONET = os.environ.get("WAVESPIKE_FAKE_PHYSIONET") if SMOKE else None   # local test: copy from a folder
STAGE_DIR = os.path.join(tempfile.gettempdir(), "wavespike_stage")               # local disk; complete files are then copied to Drive

def physionet_creds():
    if FAKE_PHYSIONET: return None
    try:
        from google.colab import userdata
        user, pw = userdata.get("PHYSIONET_USER"), userdata.get("PHYSIONET_PASS")
    except Exception as e:
        raise StopForUser("Add the Colab secrets PHYSIONET_USER and PHYSIONET_PASS (🔑 icon in the left sidebar) and enable "
                          f"notebook access for both ({type(e).__name__}).") from None
    if not user or not pw:
        raise StopForUser("The Colab secrets PHYSIONET_USER / PHYSIONET_PASS are empty.")
    return (user, pw)

def _nonempty(p): return os.path.isfile(p) and os.path.getsize(p) > 0

def _scrub(text, creds):
    text = text or ""
    for c in (creds or ()):
        if c: text = text.replace(c, "***")
    return text

def _project(base_url):                       # ".../files/<project>/<version>/" -> "<project>/<version>"
    return urllib.parse.urlparse(base_url).path.strip("/").split("/", 1)[1]

def _wget_auth(creds): return ["--user", creds[0], "--password", creds[1]] if creds else []

def physionet_listing(base_url, creds, sub=""):
    """File and folder names in one directory of a PhysioNet project, parsed from its HTML index page."""
    if FAKE_PHYSIONET:
        d = os.path.join(FAKE_PHYSIONET, _project(base_url), sub)
        return sorted(n + ("/" if os.path.isdir(os.path.join(d, n)) else "") for n in os.listdir(d))
    os.makedirs(STAGE_DIR, exist_ok=True); out = os.path.join(STAGE_DIR, "index.html")
    r = subprocess.run(["wget", "-q", "-O", out] + _wget_auth(creds) + [base_url + sub], capture_output=True, text=True)
    if r.returncode != 0:
        raise StopForUser(f"Listing {base_url + sub} failed (wget exit code {r.returncode}; 6 = authentication). Check "
                          "PHYSIONET_USER/PHYSIONET_PASS and that your account has signed this project's data use agreement. "
                          + _scrub(r.stderr, creds)[-300:])
    html = open(out, encoding="utf-8", errors="replace").read()
    prefix = urllib.parse.urlparse(base_url + sub).path
    names = set()
    for h in re.findall(r'href="([^"?#]+)"', html):
        h = urllib.parse.unquote(h)
        if h.startswith(prefix): h = h[len(prefix):]
        if not h or h.startswith(("/", "..", "#")) or "://" in h or h.startswith("mailto:"): continue
        names.add(h)
    return sorted(names)

def fetch_physionet(rel_paths, base_url, dest_root, creds, n_par=8, chunk=400, desc="files"):
    """Download base_url+rel to dest_root/rel for every rel path, skipping files that already exist.
    Uses `wget -N -c` with one URL list per process (keep-alive). Downloads go to local disk first, and a file is
    copied to Drive only once it is complete, so an interrupted run never leaves half-written files on Drive.
    Returns the paths that are still missing."""
    rel_paths = sorted(set(rel_paths))
    todo = [r for r in rel_paths if not _nonempty(os.path.join(dest_root, r))]
    if todo:
        t0 = time.time(); errors = []
        if FAKE_PHYSIONET:
            stage_root = os.path.join(FAKE_PHYSIONET, _project(base_url))
        else:
            stage_root = os.path.join(STAGE_DIR, _project(base_url).replace("/", "_")); os.makedirs(stage_root, exist_ok=True)
            cut = len(urllib.parse.urlparse(base_url).path.strip("/").split("/"))
            cmd = ["wget", "-N", "-c", "-q", "-x", "-nH", f"--cut-dirs={cut}", "-P", stage_root] + _wget_auth(creds)
        chunks = [todo[i:i+chunk] for i in range(0, len(todo), chunk)]
        def _job(ci):
            paths = chunks[ci]
            if not FAKE_PHYSIONET:
                lst = os.path.join(STAGE_DIR, f"urls_{os.getpid()}_{ci}.txt")
                with open(lst, "w") as f: f.write("\n".join(base_url + urllib.parse.quote(p) for p in paths) + "\n")
                r = subprocess.run(cmd + ["-i", lst], capture_output=True, text=True); os.remove(lst)
                if r.returncode not in (0, 8): errors.append(f"wget exit {r.returncode}: " + _scrub(r.stderr, creds)[-200:])
            for p in paths:
                s, d = os.path.join(stage_root, p), os.path.join(dest_root, p)
                if _nonempty(s):
                    os.makedirs(os.path.dirname(d) or ".", exist_ok=True)
                    shutil.copyfile(s, d + ".part"); os.replace(d + ".part", d)
                    if not FAKE_PHYSIONET: os.remove(s)
            return len(paths)
        done = 0
        with ThreadPoolExecutor(max(1, n_par)) as ex:
            for n in ex.map(_job, range(len(chunks))):
                done += n
                if len(chunks) > 1: print(f"\r  {desc}: {done}/{len(todo)} requested ({time.time()-t0:.0f}s)", end="")
        if len(chunks) > 1: print()
        for e in errors[:5]: print("  ", e)
    missing = [r for r in todo if not _nonempty(os.path.join(dest_root, r))]
    print(f"{desc}: {len(rel_paths)-len(todo)} already on disk, {len(todo)-len(missing)} downloaded, {len(missing)} missing")
    return missing

In [ ]:
ZENODO_AF = "https://zenodo.org/records/6807403/files/{}?download=1"
MIMIC_PERFORM_ZIPS = [(1, "mimic_perform_af_wfdb.zip"), (0, "mimic_perform_non_af_wfdb.zip")]

def _mimic_perform_dirs():
    raw = f"{DATA_ROOT}/raw/mimic_perform_af"; os.makedirs(raw, exist_ok=True)
    return [(label, name, extract_once(download_once(ZENODO_AF.format(name), f"{raw}/{name}"), f"{raw}/{name[:-4]}"))
            for label, name in MIMIC_PERFORM_ZIPS]

def load_mimic_perform_af(cfg):
    import wfdb
    X, y, g = [], [], []
    for label, name, d in _mimic_perform_dirs():
        heads = sorted(glob.glob(f"{d}/**/*.hea", recursive=True))
        print(f"{name}: {len(heads)} WFDB headers")
        for h in heads:
            rec = wfdb.rdrecord(h[:-4])
            names = [s.lower() for s in rec.sig_name]
            idx = [i for i, s in enumerate(names) if "ppg" in s or "pleth" in s]
            if not idx:
                print("  no PPG channel in", os.path.basename(h), rec.sig_name); continue
            sid = f"{label}_{os.path.basename(h)[:-4]}"
            for w in windows_from_signal(rec.p_signal[:, idx[0]], rec.fs, cfg.fs, cfg.win_sec):
                X.append(w); y.append(label); g.append(sid)
    return np.array(X, np.float32), np.array(y), np.array(g), ["non-AF", "AF"]

# D6: where is the MIMIC-III subject ID of a MIMIC PERform record? We search the record name, the header's record
# line and its comments (the whole .hea text) for MIMIC-III waveform-style IDs ("p000123", as in the matched
# subset folder/record names) or an explicit "subject_id". Whatever is found, or not found, is printed.
MIMIC3_ID_PATTERNS = [r"(?<![A-Za-z0-9])p(\d{6})(?![0-9])", r"subject[ _-]?id\D{0,3}(\d{1,7})"]

def mimic_perform_subject_ids():
    out = {}
    for label, name, d in _mimic_perform_dirs():
        for h in sorted(glob.glob(f"{d}/**/*.hea", recursive=True)):
            sid = f"{label}_{os.path.basename(h)[:-4]}"
            text = os.path.basename(h)[:-4] + "\n" + open(h, encoding="utf-8", errors="replace").read()
            ids = sorted({int(m.group(1)) for pat in MIMIC3_ID_PATTERNS for m in re.finditer(pat, text, re.I)})
            out[sid] = {"ids": ids, "header": text}
    return out

DEEPBEAT_MAX_PER_SUBJECT = 300

def load_deepbeat(cfg):
    # Best-effort converter: the file layout on Synapse was not verified. If it fails, run inventory(raw)
    # and adapt the key names below.
    raw = synapse_sync_once("syn21985690", f"{DATA_ROOT}/raw/deepbeat")
    files = sorted(glob.glob(f"{raw}/**/*.npz", recursive=True))
    if not files: inventory(raw); raise RuntimeError("no .npz files found; see the inventory above")
    pick = lambda d, keys: next((k for k in keys if k in d.files), None)
    X, y, g = [], [], []; rng = np.random.default_rng(0)
    for p in files:
        d = np.load(p, allow_pickle=False)          # never unpickle downloaded files
        ks, kr, kp = pick(d, ["signal", "signals", "X", "ppg"]), pick(d, ["rhythm", "label", "labels", "y"]), pick(d, ["parameters", "subject", "subjects", "ids"])
        if ks is None or kr is None or kp is None:
            inventory(raw); raise RuntimeError(f"{os.path.basename(p)}: keys {d.files}; set the signal/rhythm/subject keys in load_deepbeat")
        sig, rhy, par = d[ks], d[kr], d[kp]
        sig = sig.reshape(len(sig), -1)
        lab = rhy.argmax(1) if rhy.ndim == 2 else rhy.astype(int)
        sub = par[:, 0] if getattr(par, "ndim", 1) == 2 else par            # assumption: first column = subject id
        keep = []                                    # cap windows per subject (500K+ segments won't fit in Colab RAM)
        for s_ in np.unique(sub):
            ii = np.where(sub == s_)[0]; keep.extend(rng.choice(ii, min(len(ii), DEEPBEAT_MAX_PER_SUBJECT), replace=False))
        for i in sorted(keep):
            X.append(resample_poly(sig[i].astype(np.float64), 25, 16)); y.append(int(lab[i])); g.append(str(sub[i]))   # 32 -> 50 Hz
    print("check: DeepBeat label encoding assumed 0 = non-AF, 1 = AF. Verify on the first run.")
    return np.array(X, np.float32), np.array(y), np.array(g), ["non-AF", "AF"]

def load_uconn(syn_id, name):
    def _loader(cfg):
        raw = synapse_sync_once(syn_id, f"{DATA_ROOT}/raw/{name}")
        inventory(raw)
        raise NotImplementedError(f"{name}: downloaded to {raw}. The file layout is not public, so the converter "
                                  "isn't written yet. Share the inventory printed above and I'll write it. "
                                  "Loader code from the dataset authors: github.com/Cassey2016/UMass_Simband_Dataset")
    return _loader

def load_npz_file(cfg):
    d = np.load(NPZ_PATH, allow_pickle=True)
    X = [resample_poly(x.astype(np.float64), cfg.fs//math.gcd(NPZ_FS, cfg.fs), NPZ_FS//math.gcd(NPZ_FS, cfg.fs)) for x in d["X"]]
    names = list(d["class_names"]) if "class_names" in d.files else [str(c) for c in np.unique(d["y"])]
    return np.array(X, np.float32), d["y"].astype(int), d["subject"].astype(str), names

In [ ]:
# MIMIC-III-Ext-PPG v1.1.0 (D1–D4). Moulaeifard, Charlton & Strodthoff, Sci Data 13, 668 (2026); doi:10.13026/r6k1-xt76
EXT_BASE = "https://physionet.org/files/mimic-iii-ext-ppg/1.1.0/"
EXT_RAW = f"{DATA_ROOT}/raw/mimic3_ext_ppg"
EXT_FS_NATIVE = 125
EXT_CLASSES = ["SR", "AF", "STACH", "SBRAD"] if EXT_MULTICLASS else ["SR", "AF"]     # AF is index 1 in both
PPG_NAMES = ("pleth", "ppg")

def subject_num(s):
    """MIMIC-III subject ID as an int ("p000123", "123" and 123 -> 123), or None if the value contains no digits."""
    m = re.findall(r"\d+", str(s))
    return int(m[0]) if len(m) == 1 else None          # several digit groups = ambiguous -> None (reported as unchecked)

def ext_d1_inspect(creds):
    os.makedirs(EXT_RAW, exist_ok=True)
    listing = physionet_listing(EXT_BASE, creds)
    print("D1 top-level entries of", EXT_BASE, ":", listing)
    readmes = [n for n in listing if n.lower().startswith("readme") and not n.endswith("/")]
    problems = ([] if "metadata.csv" in listing else ["metadata.csv is not in the top-level listing"]) + \
               ([] if readmes else ["there is no README* file in the top-level listing"])
    if problems:
        raise StopForUser("; ".join(problems) + ". Check the listing above and adapt ext_d1_inspect().")
    missing = fetch_physionet(readmes + ["metadata.csv"], EXT_BASE, EXT_RAW, creds, desc="README + metadata.csv")
    if missing: raise RuntimeError(f"download failed: {missing}")
    readme = "\n\n".join(f"===== {r} =====\n" + open(f"{EXT_RAW}/{r}", encoding="utf-8", errors="replace").read() for r in readmes)
    print(readme[:20000] + ("\n[... README truncated in this printout; the full file is on Drive]" if len(readme) > 20000 else ""))
    head = pd.read_csv(f"{EXT_RAW}/metadata.csv", nrows=5000, dtype=str, low_memory=False)
    print(f"\nD1 metadata.csv: {len(head.columns)} columns (first 5000 rows inspected)")
    with pd.option_context("display.max_columns", None, "display.width", 250, "display.max_colwidth", 60):
        print(head.head(5).to_string())
    for c in head.columns:
        ex = head[c].dropna().unique()[:6]
        print(f"  {c!r:32s} non-null {head[c].notna().mean():5.1%} | unique (first 5000 rows) {head[c].nunique():6d} | e.g. {list(ex)}")
    return readme, head

def ext_d1_resolve(readme, head):
    cols = list(head.columns)
    unset = [k for k, v in EXT_SCHEMA.items() if not v]
    if unset:
        hint = {"subject_col": r"subj|patient", "label_col": r"rhythm|label|class", "path_col": r"path|file|record|segment"}
        print("\nColumns whose NAME looks related (a hint for you; the notebook does not use it):",
              {k: [c for c in cols if re.search(hint[k], c, re.I)] for k in unset})
        raise StopForUser(f"Set EXT_SCHEMA {unset} in §2 from the README and the metadata.csv columns printed above, then "
                          "run again. If a segment's file location has to be built from several columns, set path_col to "
                          "one of them and EXT_PATH_TEMPLATE accordingly. The notebook does not guess column names.")
    bad = [v for v in EXT_SCHEMA.values() if v not in cols]
    if bad: raise StopForUser(f"EXT_SCHEMA names columns that are not in metadata.csv: {bad}. Columns: {cols}")
    if len(set(EXT_SCHEMA.values())) < 3: raise StopForUser(f"EXT_SCHEMA uses one column twice: {EXT_SCHEMA}")
    print("D1 schema:", EXT_SCHEMA, "| path template:", EXT_PATH_TEMPLATE)
    for k, v in EXT_SCHEMA.items():
        hits = [l.strip() for l in readme.splitlines() if v in l][:3]
        print(f"  README lines mentioning {v!r}:", hits if hits else "NONE (check that this column means what you think)")

def ext_read_metadata():
    sc, lc, pc = EXT_SCHEMA["subject_col"], EXT_SCHEMA["label_col"], EXT_SCHEMA["path_col"]
    code2cls = {str(EXT_LABEL_CODES[c]): c for c in EXT_CLASSES}
    counts, parts = Counter(), []
    for ch in pd.read_csv(f"{EXT_RAW}/metadata.csv", usecols=[sc, lc, pc], dtype=str, chunksize=500_000):
        lab = ch[lc].fillna("<NA>").str.strip()
        counts.update(lab.value_counts().to_dict())
        m = lab.isin(list(code2cls))
        parts.append(pd.DataFrame({"subject": ch.loc[m, sc].str.strip(), "path": ch.loc[m, pc].str.strip(),
                                   "label": lab[m].map(code2cls)}))
    vc = pd.Series(counts).sort_values(ascending=False)
    print(f"D1 label column {lc!r}: {len(vc)} distinct values over {vc.sum()} rows. Top 40:\n" + vc.head(40).to_string())
    missing = [c for c in EXT_CLASSES if str(EXT_LABEL_CODES[c]) not in vc.index]
    if missing:
        raise StopForUser(f"No exact match in {lc!r} for the codes of {missing} (EXT_LABEL_CODES = {EXT_LABEL_CODES}). "
                          "Look up the code of each class in the README and set EXT_LABEL_CODES in §2.")
    for c in EXT_CLASSES:
        code = str(EXT_LABEL_CODES[c])
        compound = [v for v in vc.index if v != code and code in re.split(r"[^A-Za-z0-9]+", str(v))]
        if compound:
            raise StopForUser(f"Values {compound[:8]} contain the code {code!r} together with other text, so the label column "
                              "may hold several rhythms per segment. Decide how to treat them (edit ext_read_metadata) "
                              "and run again.")
    meta = pd.concat(parts, ignore_index=True).dropna()
    meta = meta[(meta.subject != "") & (meta.path != "")].drop_duplicates()
    conflict = meta.groupby("path").agg(nl=("label", "nunique"), ns=("subject", "nunique"))
    if (conflict.nl > 1).any() or (conflict.ns > 1).any():
        bad = conflict[(conflict.nl > 1) | (conflict.ns > 1)].head(5)
        raise StopForUser(f"Some segment paths have several labels or subjects (first: {bad.index.tolist()}). "
                          "path_col does not identify one segment; check EXT_SCHEMA.")
    print("D2 kept rows by class:", meta.label.value_counts().reindex(EXT_CLASSES).to_dict(),
          "| subjects:", meta.subject.nunique())
    return meta

def _round_robin(per_class, m):
    # Take one segment per class in turn until m segments are taken: balanced where possible, then filled up.
    out, ptr = [], {c: 0 for c in per_class}
    while len(out) < m and any(ptr[c] < len(v) for c, v in per_class.items()):
        for c in EXT_CLASSES:
            if c in per_class and ptr[c] < len(per_class[c]) and len(out) < m:
                out.append(per_class[c][ptr[c]]); ptr[c] += 1
    return out

def ext_select(meta, exclude_ids):
    meta = meta.copy()
    meta["sid"] = meta.subject.map(subject_num)
    if exclude_ids is not None:
        if meta.sid.isna().any():
            print(f"⚠ D6: {meta.sid.isna().sum()} rows have a subject value without digits; they can't be checked for overlap")
        ovl = meta.sid.isin(set(exclude_ids))
        print(f"D6 overlap with MIMIC PERform AF: {meta.subject[ovl].nunique()} Ext-PPG subjects ({int(ovl.sum())} "
              f"segments of the kept classes) -> removed before selection: {sorted(meta.subject[ovl].unique().tolist())}")
        meta = meta[~ovl]
    has_af = meta.groupby("subject")["label"].apply(lambda s: bool((s == "AF").any()))
    af_s, non_s = sorted(has_af.index[has_af]), sorted(has_af.index[~has_af])
    rng = np.random.default_rng(EXT_SELECT_SEED)
    n_af = min(len(af_s), N_SUBJECTS // 2); n_non = min(len(non_s), N_SUBJECTS - n_af); n_af = min(len(af_s), N_SUBJECTS - n_non)
    chosen = sorted(rng.choice(af_s, n_af, replace=False).tolist() + rng.choice(non_s, n_non, replace=False).tolist())
    print(f"D3a subjects available: {len(af_s)} with AF, {len(non_s)} without -> picked {n_af} with AF + {n_non} without "
          f"(seed {EXT_SELECT_SEED})")
    rows = []
    for s, g in meta[meta.subject.isin(chosen)].sort_values(["subject", "path"]).groupby("subject", sort=True):
        per = {c: g[g.label == c].path.tolist() for c in EXT_CLASSES if (g.label == c).any()}
        per = {c: [v[i] for i in rng.permutation(len(v))] for c, v in per.items()}
        lab_of = dict(zip(g.path, g.label))
        rows += [(s, p, lab_of[p]) for p in _round_robin(per, MAX_SEG_PER_SUBJECT)]
    sel = pd.DataFrame(rows, columns=["subject", "path", "label"])
    def _rec(r):
        p = EXT_PATH_TEMPLATE.format(path=r.path, subject=r.subject)
        return re.sub(r"\.(hea|dat)$", "", p).lstrip("/")
    sel["record"] = [_rec(r) for r in sel.itertuples()]
    print(f"D3b selected {len(sel)} segments:", sel.label.value_counts().reindex(EXT_CLASSES).to_dict())
    return sel[["subject", "record", "label", "path"]]

def ext_tag(exclude_ids):
    key = json.dumps([N_SUBJECTS, MAX_SEG_PER_SUBJECT, EXT_SELECT_SEED, EXT_CLASSES, EXT_SCHEMA, EXT_PATH_TEMPLATE,
                      {c: str(EXT_LABEL_CODES[c]) for c in EXT_CLASSES}, sorted(exclude_ids or [])], sort_keys=True)
    return f"n{N_SUBJECTS}_m{MAX_SEG_PER_SUBJECT}_s{EXT_SELECT_SEED}_{'mc' if EXT_MULTICLASS else 'bin'}_" + \
           hashlib.sha1(key.encode()).hexdigest()[:8]

def pick_ppg_channel(sig_names):
    low = [str(s).lower().strip() for s in sig_names]
    exact = [i for i, s in enumerate(low) if s in PPG_NAMES]
    part = [i for i, s in enumerate(low) if any(p in s for p in PPG_NAMES)]
    return exact[0] if exact else (part[0] if part else None)

def ext_verify_layout(man, creds, n=3):
    import wfdb
    rows = man.head(n)
    missing = fetch_physionet([r + ".hea" for r in rows.record], EXT_BASE, EXT_RAW, creds, desc="layout check headers")
    if missing:
        raise StopForUser(f"Headers not found on PhysioNet: {missing}. path_col + EXT_PATH_TEMPLATE do not give the file "
                          "location; check the folder layout in the README and set EXT_PATH_TEMPLATE in §2.")
    for r in rows.record:
        hd = wfdb.rdheader(os.path.join(EXT_RAW, r))
        segs = getattr(hd, "seg_name", None)
        print(f"  layout check {r}: fs={hd.fs} sig_len={hd.sig_len} ({hd.sig_len/hd.fs:.1f} s) sig_name={hd.sig_name} "
              f"files={sorted(set(hd.file_name or []))} comments={(hd.comments or [])[:3]}")
        if segs: raise StopForUser(f"{r} is a multi-segment record ({segs[:3]}...); the loader expects one record per segment.")
        if abs(float(hd.fs) - EXT_FS_NATIVE) > 1e-6: raise StopForUser(f"{r}: fs = {hd.fs}, expected {EXT_FS_NATIVE} Hz")
        if pick_ppg_channel(hd.sig_name) is None: raise StopForUser(f"{r}: no PLETH/PPG channel in {hd.sig_name}")
        if not 25 <= hd.sig_len / hd.fs <= 31:
            raise StopForUser(f"{r}: record is {hd.sig_len/hd.fs:.1f} s long, expected a 30 s segment. The metadata may index "
                              "into longer records (start/end columns?). Adapt ext_preprocess before going on.")

def ext_download(man, man_path, creds):
    import wfdb
    miss_h = set(fetch_physionet([r + ".hea" for r in man.record], EXT_BASE, EXT_RAW, creds, n_par=EXT_DOWNLOAD_PARALLEL,
                                 desc="segment headers"))
    sig_files, bad = set(), []
    for r in man.record:
        if r + ".hea" in miss_h: continue
        try:
            hd = wfdb.rdheader(os.path.join(EXT_RAW, r))
            sig_files.update(os.path.join(os.path.dirname(r), f) for f in set(hd.file_name or []))
        except Exception as e:
            bad.append((r, type(e).__name__))
    if bad: print(f"⚠ {len(bad)} unreadable headers, e.g. {bad[:3]}")
    miss_s = set(fetch_physionet(sorted(sig_files), EXT_BASE, EXT_RAW, creds, n_par=EXT_DOWNLOAD_PARALLEL, desc="signal files"))
    def _ok(r):
        if r + ".hea" in miss_h or r in dict(bad): return False
        hd = wfdb.rdheader(os.path.join(EXT_RAW, r))
        return all(os.path.join(os.path.dirname(r), f) not in miss_s for f in set(hd.file_name or []))
    man["downloaded"] = [_ok(r) for r in man.record]
    man.to_csv(man_path, index=False)
    print(f"downloaded {int(man.downloaded.sum())}/{len(man)} selected segments | manifest: {man_path}")
    return man

def clean_segment(sig, fs, min_valid_s=25.0, flat_run_s=1.0):
    """(signal or None, reason). Invalid samples: NaN/inf, or inside a run of identical values lasting >= flat_run_s."""
    sig = np.asarray(sig, np.float64)
    if not np.all(np.isfinite(sig)): return None, "nan"
    if len(sig) == 0 or sig.std() < 1e-6: return None, "flat"
    change = np.flatnonzero(np.diff(sig) != 0) + 1
    starts, ends = np.r_[0, change], np.r_[change, len(sig)]
    flat = int(sum(e - s for s, e in zip(starts, ends) if e - s >= flat_run_s*fs))
    if (len(sig) - flat) / fs < min_valid_s: return None, "<25s valid"
    return sig, "ok"

def ext_preprocess(man, cfg):
    import wfdb
    n_nat = int(round(cfg.win_sec * EXT_FS_NATIVE)); g = math.gcd(EXT_FS_NATIVE, cfg.fs)
    up, down = cfg.fs // g, EXT_FS_NATIVE // g                     # 125 -> 50 Hz: up=2, down=5
    X, y, grp, recs, reasons = [], [], [], [], []
    for i, r in enumerate(man.itertuples()):
        if not r.downloaded: reasons.append("not downloaded"); continue
        try:
            rec = wfdb.rdrecord(os.path.join(EXT_RAW, r.record))
        except Exception as e:
            reasons.append("unreadable"); continue
        ch = pick_ppg_channel(rec.sig_name)
        if ch is None: reasons.append("no PPG channel"); continue
        if abs(float(rec.fs) - EXT_FS_NATIVE) > 1e-6: reasons.append(f"fs {rec.fs}"); continue
        sig, why = clean_segment(rec.p_signal[:n_nat, ch], EXT_FS_NATIVE)
        if sig is None: reasons.append(why); continue
        why = "ok"
        if len(sig) < n_nat: sig = np.pad(sig, (0, n_nat - len(sig)), mode="edge"); why = "ok (edge-padded to 30 s)"
        X.append(resample_poly(sig, up, down).astype(np.float32)); y.append(EXT_CLASSES.index(r.label))
        grp.append(str(r.subject)); recs.append(r.record); reasons.append(why)
        if (i + 1) % 2000 == 0: print(f"\r  preprocessed {i+1}/{len(man)}", end="")
    print()
    man["status"] = reasons
    print("D4 segment status:", pd.Series(reasons).value_counts().to_dict())
    return np.stack(X), np.array(y), np.array(grp), np.array(recs)

def load_mimic3_ext_ppg(cfg, exclude_ids):
    tag = ext_tag(exclude_ids)
    cache = f"{DATA_ROOT}/processed/mimic3_ext_ppg_{cfg.fs}hz_{cfg.win_sec}s{'_mc' if EXT_MULTICLASS else ''}.npz"
    if os.path.exists(cache):
        d = np.load(cache, allow_pickle=False)
        if str(d["tag"]) == tag:
            print("loaded cache", cache, "| selection", tag)
            return d["X"], d["y"], d["subject"], d["record"], [str(c) for c in d["class_names"]]
        print(f"cache {cache} was built with another selection ({d['tag']}); rebuilding for {tag}")
    creds = physionet_creds()
    man_path = f"{EXT_RAW}/manifest_{tag}.csv"
    if os.path.exists(man_path):
        man = pd.read_csv(man_path, dtype={"subject": str, "record": str, "label": str, "path": str})
        print("D3 reusing manifest", man_path, f"({len(man)} segments)")
    else:
        readme, head = ext_d1_inspect(creds)
        ext_d1_resolve(readme, head)
        man = ext_select(ext_read_metadata(), exclude_ids)
        ext_verify_layout(man, creds)
        man.to_csv(man_path, index=False); print("D3 manifest written:", man_path)
    man = ext_download(man, man_path, creds)
    X, y, g, recs = ext_preprocess(man, cfg)
    man.to_csv(man_path, index=False)
    np.savez_compressed(cache, X=X, y=y, subject=g, record=recs, class_names=np.array(EXT_CLASSES), tag=np.array(tag))
    print("cached to", cache)
    return X, y, g, recs, EXT_CLASSES

In [ ]:
# Synthetic generator (smoke test only; heuristic, not a physiological model)
def _beat(n, shape):
    t = np.linspace(0, 1, n, endpoint=False); a1, m1, s1, a2, m2, s2 = shape
    return a1*np.exp(-0.5*((t-m1)/s1)**2) + a2*np.exp(-0.5*((t-m2)/s2)**2)

def synth_window(label, rng, subj, fs=50, sec=30):
    N = fs*sec; out = []; total = 0; hr = subj["hr"]*rng.uniform(0.93, 1.07); rr0 = 60.0/hr
    ect = set(rng.choice(np.arange(3, 3+int(sec/rr0)-4), size=rng.integers(3, 7), replace=False)) if label == 2 else set()
    k = 0; pause = False
    while total < N + fs:
        if label == 0:   rr = rr0*(1 + 0.04*np.sin(2*np.pi*0.25*total/fs) + 0.02*rng.standard_normal())
        elif label == 1: rr = rr0*np.clip(1 + 0.22*rng.standard_normal(), 0.45, 1.8)
        else:
            if k in ect:  rr = rr0*rng.uniform(0.55, 0.75); pause = True
            elif pause:   rr = rr0*rng.uniform(1.25, 1.45); pause = False
            else:         rr = rr0*(1 + 0.02*rng.standard_normal())
        n = max(int(rr*fs), 8); amp = np.clip(rr/rr0, 0.4, 1.4)**(0.8 if label else 0.2)
        out.append(amp*_beat(n, subj["shape"])); total += n; k += 1
    x = np.concatenate(out)[:N]; tt = np.arange(N)/fs
    x = x + 0.3*np.sin(2*np.pi*rng.uniform(0.05, 0.3)*tt + rng.uniform(0, 6.3)) + 0.05*rng.standard_normal(N)
    if rng.random() < 0.3:
        s = rng.integers(0, N-fs*3); x[s:s+fs*2] += rng.normal(0, 0.6, fs*2).cumsum()*0.05
    return x

def load_synthetic(cfg, n_subjects=48, win_per_subject=25, seed=123):
    rng = np.random.default_rng(seed); X, y, g = [], [], []
    for s in range(n_subjects):
        subj = {"hr": rng.uniform(55, 100), "shape": (1.0, rng.uniform(.18, .3), rng.uniform(.06, .1), rng.uniform(.25, .6), rng.uniform(.45, .6), rng.uniform(.08, .14))}
        p = rng.dirichlet([1.0, 1.0, 1.0])
        for _ in range(win_per_subject):
            lab = int(rng.choice(3, p=p)); X.append(synth_window(lab, rng, subj, cfg.fs, cfg.win_sec)); y.append(lab); g.append(str(s))
    return np.array(X, np.float32), np.array(y), np.array(g), ["NSR", "AF", "PAC/PVC"]

# ---- Zenodo 10.5281/zenodo.23597128: long-term wrist PPG (100 Hz, green LED) with ECG-derived beat-by-beat AF annotations ----
# Open, no account; CC BY-NC-SA 4.0 (non-commercial, share-alike). Only the PPG is used as model input; the ECG files only provide labels.
# Files are read from $DATA_ROOT/raw/zenodo_23597128 (download them first; this loader never downloads 60 GB by itself).
ZEN_PPG_FS, ZEN_ECG_FS = 100, 500
ZEN_MAX_PER_CLASS = int(os.environ.get("WAVESPIKE_ZEN_MAX_PER_CLASS", 300))   # windows per subject and class (random, seeded)
ZEN_MIN_BEATS = 12                     # a window needs >= 12 annotated beats to get a label
ZEN_AF_MIN_FRAC = 0.95                 # AF window: >= 95 % of its beats annotated AF; non-AF window: 0 AF beats; mixed windows are skipped

def _zen_text(h5, ds):
    return "".join(chr(int(c)) for c in np.array(ds).ravel())

def _zen_seconds(day, tod):            # ("02", "08:14:41") -> seconds since the first monitoring day, 00:00
    h, m, s_ = (int(v) for v in tod.split(":")); return (int(day) - 1)*86400 + h*3600 + m*60 + s_

def load_zenodo_wrist_af(cfg):
    import h5py
    raw = f"{DATA_ROOT}/raw/zenodo_23597128"; rng = np.random.default_rng(0)
    subs = sorted(f[:3] for f in os.listdir(raw) if f.endswith("_PPG.mat") and os.path.exists(f"{raw}/{f[:3]}_ECG.mat"))
    if not subs: raise StopForUser(f"No <id>_PPG.mat / <id>_ECG.mat pairs in {raw}. Download them from https://zenodo.org/records/23597128 first.")
    n = int(cfg.win_sec*ZEN_PPG_FS); g_ = math.gcd(ZEN_PPG_FS, cfg.fs); up, down = cfg.fs//g_, ZEN_PPG_FS//g_
    X, y, g = [], [], []
    for sid in subs:
        e = h5py.File(f"{raw}/{sid}_ECG.mat"); p = h5py.File(f"{raw}/{sid}_PPG.mat")      # an unreadable/truncated file raises: no silent skipping
        af = np.array(e["AF_annotation"]).ravel(); bt = np.array(e["QRSindex"]).ravel()[1:len(af) + 1]/ZEN_ECG_FS
        e0 = _zen_seconds(_zen_text(e, e["recording_startday"]), _zen_text(e, e["recording_starttime"]))
        refs = p["PPG_GREEN"]; cand = []                                                # (segment, window index, label)
        for k in range(refs.shape[0]):
            ds = p[refs[k, 0]]; s0 = _zen_seconds(_zen_text(p, p[p["recording_startday"][k, 0]]), _zen_text(p, p[p["recording_starttime"][k, 0]])) - e0
            for i in range(ds.shape[-1]//n):
                t0 = s0 + i*cfg.win_sec; lo, hi = np.searchsorted(bt, [t0, t0 + cfg.win_sec]); b = af[lo:hi]
                if len(b) < ZEN_MIN_BEATS or np.isnan(b).any(): continue
                fr = float(b.mean())
                if fr >= ZEN_AF_MIN_FRAC: cand.append((k, i, 1))
                elif fr == 0.0: cand.append((k, i, 0))
        pick = []
        for lab in (0, 1):
            c = [t for t in cand if t[2] == lab]
            if c: pick += [c[j] for j in rng.choice(len(c), min(ZEN_MAX_PER_CLASS, len(c)), replace=False)]
        by_seg = {}
        for k, i, lab in pick: by_seg.setdefault(k, []).append((i, lab))
        kept = Counter()
        for k, items in by_seg.items():
            sig = np.array(p[refs[k, 0]]).ravel()
            for i, lab in items:
                w, why = clean_segment(sig[i*n:(i + 1)*n], ZEN_PPG_FS)
                if w is None: continue
                X.append(resample_poly(w, up, down).astype(np.float32)); y.append(lab); g.append(sid); kept[lab] += 1
        print(f"zenodo_wrist_af subject {sid}: {len(cand)} labelled windows, kept non-AF {kept[0]} / AF {kept[1]}")
    return np.array(X, np.float32), np.array(y), np.array(g), ["non-AF", "AF"]

LOADERS = {"mimic_perform_af": (load_mimic_perform_af, 30), "deepbeat": (load_deepbeat, 25),
           "uconn_simband": (load_uconn("syn25005551", "uconn_simband"), 30),
           "uconn_mimic3": (load_uconn("syn25005553", "uconn_mimic3"), 30),
           "pulsewatch": (load_uconn("syn23565056", "pulsewatch"), 30),
           "npz": (load_npz_file, 30), "synthetic": (load_synthetic, 30),
           "mimic3_ext_ppg": (None, 30), "zenodo_wrist_af": (load_zenodo_wrist_af, 30)}            # handled by load_mimic3_ext_ppg (needs the D6 exclusion list)

In [ ]:
# Load (Drive cache when available) -> D6 overlap check -> normalise -> D5 subject-wise folds
def load_cached(name, loader, cfg, **kw):
    cache = f"{DATA_ROOT}/processed/{name}_{cfg.fs}hz_{cfg.win_sec}s.npz"
    if os.path.exists(cache) and name != "synthetic":
        d = np.load(cache, allow_pickle=True); print("loaded cache", cache)
        return d["X"], d["y"], d["subject"], [str(c) for c in d["class_names"]]
    X_, y_, g_, names = loader(cfg, **kw)
    if name != "synthetic":
        np.savez_compressed(cache, X=X_, y=y_, subject=g_, class_names=np.array(names)); print("cached to", cache)
    return X_, y_, g_, names

def normalize_windows(X):
    mu = X.mean(1, keepdims=True); sd = X.std(1, keepdims=True) + 1e-6
    return ((X - mu)/sd).astype(np.float32)

# ---- D6 part 1: MIMIC-III subject IDs of the external set (must run before Ext-PPG subjects are picked) ----
EXCLUDE_IDS, EXTERNAL_OVERLAP_STATUS = None, "n/a (no external set)"
XE = yE = gE = None
if EXTERNAL_DATASET == "mimic_perform_af":
    _ids = mimic_perform_subject_ids()
    _with = {k: v["ids"] for k, v in _ids.items() if v["ids"]}
    _first = next(iter(_ids.values()))["header"]
    print("D6 first MIMIC PERform AF header (searched for an ID):\n   " + "\n   ".join(_first.splitlines()[:12]))
    print(f"D6 MIMIC-III subject ID found in {len(_with)}/{len(_ids)} MIMIC PERform AF records, e.g. "
          f"{dict(list(_with.items())[:5])}")
    EXCLUDE_IDS = sorted({i for v in _with.values() for i in v})
    if len(_with) == len(_ids):
        EXTERNAL_OVERLAP_STATUS = "verified"
    else:
        EXTERNAL_OVERLAP_STATUS = (f"UNVERIFIED: no MIMIC-III subject ID in the name or header of {len(_ids)-len(_with)}"
                                   f"/{len(_ids)} MIMIC PERform AF records")
        print("⚠" * 3, "D6:", EXTERNAL_OVERLAP_STATUS, "-> overlap with Ext-PPG can't be fully excluded. External "
              "results are labelled UNVERIFIED and kept out of the main table.")
    XE, yE, gE, CLASS_NAMES_E = load_cached("mimic_perform_af", load_mimic_perform_af, CFG)
    XE = normalize_windows(XE)
    print(f"external MIMIC PERform AF: {XE.shape[0]} windows, {len(np.unique(gE))} subjects, classes "
          f"{dict(zip(CLASS_NAMES_E, np.bincount(yE).tolist()))}")

# ---- training data ----
loader, CFG.win_sec = LOADERS[DATASET]
if DATASET == "mimic3_ext_ppg":
    X, y, groups, RECORD_IDS, CLASS_NAMES = load_mimic3_ext_ppg(CFG, EXCLUDE_IDS)
else:
    X, y, groups, CLASS_NAMES = load_cached(DATASET, loader, CFG, **(SYNTH_KW if DATASET == "synthetic" else {}))
    RECORD_IDS = np.array([f"{g}#{i}" for i, g in enumerate(groups)])
groups = np.asarray(groups).astype(str)

# ---- D6 part 2: assert there is no overlap left in the training data ----
if EXCLUDE_IDS is not None:
    _num = np.array([subject_num(g) for g in groups], dtype=object)
    _ov = np.array([n is not None and n in set(EXCLUDE_IDS) for n in _num])
    print(f"D6 Ext-PPG subjects overlapping MIMIC PERform AF in the training data: {len(set(groups[_ov]))} -> dropped")
    if _ov.any(): X, y, groups, RECORD_IDS = X[~_ov], y[~_ov], groups[~_ov], RECORD_IDS[~_ov]
    if any(n is None for n in _num):
        EXTERNAL_OVERLAP_STATUS = "UNVERIFIED: some Ext-PPG subject IDs have no single number"
        print("⚠ D6:", EXTERNAL_OVERLAP_STATUS)

X = normalize_windows(X); CFG.n_classes = len(CLASS_NAMES); T_WIN = X.shape[1]; CFG.win_sec = T_WIN // CFG.fs
AF_IDX = CLASS_NAMES.index("AF") if "AF" in CLASS_NAMES else None
print(f"{DATASET}: X {X.shape} | subjects {len(np.unique(groups))} | segments per class "
      f"{dict(zip(CLASS_NAMES, np.bincount(y, minlength=CFG.n_classes).tolist()))}")

def subject_strat_labels(y, groups, af_idx):
    # D5: stratify on the SUBJECT's AF status (any AF segment). Without an AF class: the subject's majority label.
    u, inv = np.unique(groups, return_inverse=True)
    if af_idx is not None:
        s = np.zeros(len(u), int); np.maximum.at(s, inv, (y == af_idx).astype(int)); return s[inv]
    cnt = np.zeros((len(u), y.max()+1), int); np.add.at(cnt, (inv, y), 1); return cnt.argmax(1)[inv]

STRAT = subject_strat_labels(y, groups, AF_IDX)
if AF_IDX is not None:
    _sa = pd.Series(STRAT).groupby(groups).first()
    print(f"subjects with any AF segment: {int(_sa.sum())} / {len(_sa)}")
DATA_FP = hashlib.sha1((DATASET + "|" + "|".join(RECORD_IDS.astype(str)) + "|" + y.tobytes().hex()).encode()).hexdigest()

def make_folds(y, groups, n_folds, seed=0):
    # Stratified subject-wise outer folds; inside each training part, one stratified group fold becomes validation.
    # `y` here is the stratification label (the subject's AF status), not the segment label.
    outer = StratifiedGroupKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    folds = []
    for trva, te in outer.split(np.zeros(len(y)), y, groups):
        inner = StratifiedGroupKFold(n_splits=max(3, n_folds-1), shuffle=True, random_state=seed+1)
        tr_i, va_i = next(inner.split(np.zeros(len(trva)), y[trva], groups[trva]))
        tr, va = trva[tr_i], trva[va_i]
        assert not (set(groups[tr]) & set(groups[te])) and not (set(groups[va]) & set(groups[te])) and not (set(groups[tr]) & set(groups[va]))
        folds.append((tr, va, te))
    assert np.array_equal(np.sort(np.concatenate([f[2] for f in folds])), np.arange(len(y))), "test folds must partition the data"
    return folds

FOLDS_BY_CV = {cs: make_folds(STRAT, groups, CFG.n_folds, seed=cs) for cs in CV_SEEDS}
FOLDS = FOLDS_BY_CV[CV_SEEDS[0]]
for k, (tr, va, te) in enumerate(FOLDS):
    print(f"fold {k}: train {len(tr)} ({len(set(groups[tr]))} subj) | val {len(va)} ({len(set(groups[va]))}) | test {len(te)} "
          f"({len(set(groups[te]))}) | test classes {np.bincount(y[te], minlength=CFG.n_classes)}")

fig, ax = plt.subplots(CFG.n_classes, 1, figsize=(12, 1.8*CFG.n_classes), sharex=True)
for c in range(CFG.n_classes):
    i = np.where(y == c)[0][0]; ax[c].plot(np.arange(T_WIN)/CFG.fs, X[i], lw=.8); ax[c].set_ylabel(CLASS_NAMES[c])
ax[-1].set_xlabel("s"); plt.tight_layout(); plt.show()

## §3 Learnable causal wavelet bank (per level) + learnable hard threshold
- **Filters:** level *j* learns its own low-pass filter h₀⁽ʲ⁾. The high-pass filter is its CQF partner, h₁⁽ʲ⁾[k] = (−1)ᵏ h₀⁽ʲ⁾[L−1−k], as in DeSpaWN's "one kernel per level, the others from CQF".
- **Transform:** undecimated and causal, with filters dilated by 2^(j−1) (à trous, Quotb et al. 2011). The output at time *t* uses only past samples.
- **Wavelet loss:** averaged over levels. It is double-shift orthonormality of h₀ (equivalent to perfect reconstruction for a CQF bank) plus a zero-mean h₁ (one vanishing moment). This soft constraint follows Wolter et al. 2020; DeSpaWN does not impose orthonormality.
- **Hard threshold (DeSpaWN eq. 5):** HT(x) = x·[σ(−α(x+b⁻)) + σ(α(x−b⁺))], applied to each band, with learnable b⁺ and b⁻.
  - **v3 default `ht_nonneg=True` (P1.6):** b = softplus(raw), with raw initialised at inv_softplus(0.02). Then b ≥ 0, so the gain at x → 0, σ(−αb⁻)+σ(−αb⁺), is **≤ 1**: the layer can only shrink small coefficients.
  - In v2 the unconstrained b went negative in d1–d4, giving a gain of 1.09–1.42 at x → 0 (amplification).
  - `ht_nonneg=False` keeps the unconstrained DeSpaWN version (b starts at 0, an exact identity).
- **Dead bands (P1.7):** `drop_bands=[0, 1]` removes d1 and d2 (encoder rate 0.001 in v2) after the transform. The remaining bands all lie below 6.25 Hz, so `snn_decim=2` (keep every 2nd sample, i.e. 25 Hz) does not alias. The readout warm-up becomes ⌈w₀ / snn_decim⌉ decimated steps.

In [ ]:
DB4 = [-0.010597401784997278, 0.032883011666982945, 0.030841381835986965, -0.18703481171888114,
       -0.02798376941698385, 0.6308807679295904, 0.7148465705525415, 0.23037781330885523]

def inv_softplus(v): return math.log(math.expm1(v))
def logit(p):        return math.log(p/(1-p))

def init_lowpass(name, L):
    if name == "db4": assert L == 8; return np.array(DB4)
    if name == "haar": h = np.zeros(L); h[-2:] = 1/np.sqrt(2); return h
    raise ValueError(name)

def qmf_highpass(h0):
    L = h0.shape[-1]
    sign = torch.tensor([(-1.0)**k for k in range(L)], dtype=h0.dtype, device=h0.device)
    return sign*torch.flip(h0, [-1])

class LearnableSWT(nn.Module):
    def __init__(self, levels, h0_init, learnable=True, per_level=True):
        super().__init__()
        self.levels, self.L, self.per_level = levels, len(h0_init), per_level
        h0 = torch.tensor(np.tile(np.asarray(h0_init, np.float64), (levels if per_level else 1, 1)), dtype=torch.float32)
        if learnable: self.h0 = nn.Parameter(h0)
        else:         self.register_buffer("h0", h0)
        self.warmup = sum((2**j)*(self.L-1) for j in range(levels))

    def lowpass(self, j): return self.h0[j if self.per_level else 0]

    def forward(self, x):                                     # (B, T) -> (B, levels+1, T)
        a, outs = x.unsqueeze(1), []
        for j in range(self.levels):
            h0 = self.lowpass(j); h1 = qmf_highpass(h0); d = 2**j
            w = torch.stack([h0, h1]).flip(-1).unsqueeze(1)   # flip: conv1d is cross-correlation
            y_ = F.conv1d(F.pad(a, (d*(self.L-1), 0)), w, dilation=d) / math.sqrt(2.0)
            a = y_[:, :1]; outs.append(y_[:, 1:2])
        outs.append(a)
        return torch.cat(outs, 1)

def wavelet_loss(H0):
    H0 = H0 if H0.dim() == 2 else H0.unsqueeze(0); L = H0.shape[-1]; tot = 0.0
    for h0 in H0:
        h1 = qmf_highpass(h0)
        ac = F.conv1d(h0.view(1, 1, -1), h0.view(1, 1, -1), padding=L-1).view(-1)
        even = ac[L-1::2]; tgt = torch.zeros_like(even); tgt[0] = 1.0
        tot = tot + ((even-tgt)**2).sum() + h1.sum()**2
    return tot/H0.shape[0]

class LearnableHT(nn.Module):
    # Michau, Frusque & Fink 2021, eq. 5: HT(x) = x·[σ(−α(x+b⁻)) + σ(α(x−b⁺))]; one (b⁺, b⁻) pair per band.
    # nonneg=True (P1.6): b = softplus(raw) >= 0, so the gain at x -> 0 is σ(−αb⁻)+σ(−αb⁺) <= 1 (shrink only).
    def __init__(self, C, alpha=10.0, nonneg=True, init=0.02):
        super().__init__(); self.alpha, self.nonneg = alpha, nonneg
        if nonneg:
            self.b_pos_raw = nn.Parameter(torch.full((C,), inv_softplus(init)))
            self.b_neg_raw = nn.Parameter(torch.full((C,), inv_softplus(init)))
        else:                                                 # unconstrained DeSpaWN version, starts as an identity
            self.b_pos = nn.Parameter(torch.zeros(C)); self.b_neg = nn.Parameter(torch.zeros(C))
    def thresholds(self):
        if self.nonneg: return F.softplus(self.b_pos_raw), F.softplus(self.b_neg_raw)
        return self.b_pos, self.b_neg
    def gain_at_zero(self):
        bp, bn = self.thresholds(); return torch.sigmoid(-self.alpha*bn) + torch.sigmoid(-self.alpha*bp)
    def forward(self, x):                                     # (B, C, T)
        bp, bn = self.thresholds(); bp, bn = bp.view(1, -1, 1), bn.view(1, -1, 1)
        return x*(torch.sigmoid(-self.alpha*(x + bn)) + torch.sigmoid(self.alpha*(x - bp)))

print("db4 wavelet loss (≈0):", float(wavelet_loss(torch.tensor(DB4))))
_ht = LearnableHT(1, 10.0, True, 0.02)
with torch.no_grad(): print("HT (ht_nonneg) at init: b = %.4f, gain at x→0 = %.3f (≤ 1)" % (float(_ht.thresholds()[0]), float(_ht.gain_at_zero())))

## §4 Spiking network (and the non-spiking twin)
- **Encoder:** u⁺ ← β·u⁺ + g·b and u⁻ ← β·u⁻ − g·b. A neuron spikes when u⁺ ≥ θ⁺ or u⁻ ≥ θ⁻ and then resets by subtraction. The gain g, leak β and thresholds θ⁺, θ⁻ are learned per band.
- **Hidden layer:** recurrent LIF neurons with a learnable leak per neuron and threshold 1. With `spike_dropout > 0`, hidden spikes are dropped (inverted dropout) in training only, both in the recurrence and at the readout.
- **Readout:** leaky integrators averaged after the filter warm-up, computed in closed form.
- **Leak initialisation:** with `snn_decim = n`, every leak β is initialised to β₀ⁿ (0.9ⁿ for the encoder, 0.95ⁿ for hidden and readout), so time constants in seconds match the 50 Hz model. With n = 1 this is the same as v2.
- **Training:** surrogate gradients (fast sigmoid, Zenke & Ganguli 2018).
- **`arch="gru"` (P0.4b):** same wavelet/HT/gain front end, then a GRU over the bands. Its hidden size is chosen so that the total parameter count matches the SNN's. Its readout is a linear layer on the mean GRU state after the warm-up. The Pi export supports only the SNN.
- **`use_wavelet=False` (P0.4c):** the normalised raw signal goes straight into a single ± encoder pair (no wavelet, no HT, no warm-up).

The Pi version (§10) runs the same equations one sample at a time.

In [ ]:
class SpikeFn(torch.autograd.Function):
    @staticmethod
    def forward(ctx, v, slope):
        ctx.save_for_backward(v); ctx.slope = slope; return (v >= 0).to(v.dtype)
    @staticmethod
    def backward(ctx, g):
        (v,) = ctx.saved_tensors; return g/(ctx.slope*v.abs() + 1.0)**2, None
spike = SpikeFn.apply

def readout_weights(bo, T, w0):
    # Exact: mean_{t>=w0} vo[t] with vo[t] = bo*vo[t-1] + z[t]  ==  sum_s z[s]*W[s] / (T-w0)
    s = torch.arange(T, device=bo.device, dtype=bo.dtype).unsqueeze(1)
    tail = (1 - bo**(T - torch.clamp(s, min=w0))) / (1 - bo)
    lead = bo**torch.clamp(w0 - s, min=0)
    return lead*tail                                          # (T, K)

BAND_NAMES = lambda levels: [f"d{j+1}" for j in range(levels)] + [f"a{levels}"]

class WaveFront(nn.Module):
    """Shared front end: SWT -> keep bands -> HT -> per-band gain -> decimation. Parameter names match v2."""
    def _build_frontend(self, cfg):
        if cfg.use_wavelet:
            self.swt = LearnableSWT(cfg.levels, init_lowpass(cfg.wavelet_init, cfg.taps), cfg.learn_wavelet, cfg.per_level)
            n_all, w0, names = cfg.levels + 1, self.swt.warmup, BAND_NAMES(cfg.levels)
        else:
            assert not cfg.drop_bands, "drop_bands needs the wavelet"
            self.swt, n_all, w0, names = None, 1, 0, ["raw"]
        assert all(0 <= b < n_all for b in cfg.drop_bands), f"drop_bands must be in 0..{n_all-1}"
        self.keep = [c for c in range(n_all) if c not in set(cfg.drop_bands)]
        assert self.keep, "all bands dropped"
        self.band_names = [names[c] for c in self.keep]
        self.C, self.decim = len(self.keep), int(cfg.snn_decim)
        assert self.decim >= 1
        self.w0 = -(-w0 // self.decim)                        # warm-up in (decimated) SNN steps: ceil(w0 / decim)
        self.ht = LearnableHT(self.C, cfg.ht_alpha, cfg.ht_nonneg, cfg.ht_init) if (cfg.use_ht and cfg.use_wavelet) else None
        self.enc_gain = nn.Parameter(torch.ones(self.C))

    def frontend(self, x):                                    # (B, T) -> (B, C, ceil(T/decim))
        if self.swt is not None:
            bands = self.swt(x)
            if self.C < bands.shape[1]:                       # slicing + cat: deterministic backward (no index_add)
                bands = torch.cat([bands[:, c:c+1] for c in self.keep], 1)
        else:
            bands = x.unsqueeze(1)
        if self.ht is not None: bands = self.ht(bands)
        bands = bands*self.enc_gain.view(1, -1, 1)
        if self.decim > 1: bands = bands[:, :, ::self.decim]
        return bands

class WaveSpikeNet(WaveFront):
    def __init__(self, cfg):
        super().__init__(); self.cfg = cfg; self._build_frontend(cfg)
        C, H, K, n = self.C, cfg.hidden, cfg.n_classes, self.decim
        self.enc_beta_raw = nn.Parameter(torch.full((C,), logit(0.9**n)))
        self.enc_th_pos_raw = nn.Parameter(torch.full((C,), inv_softplus(1.0)))
        self.enc_th_neg_raw = nn.Parameter(torch.full((C,), inv_softplus(1.0)))
        self.fc_in = nn.Linear(2*C, H)
        self.fc_rec = nn.Linear(H, H, bias=False); nn.init.orthogonal_(self.fc_rec.weight, gain=0.5)
        if cfg.tau_het:        # heterogeneous time constants (seconds): one SNN step lasts n/fs s, so beta = exp(-(n/fs)/tau)
            taus = np.exp(np.linspace(np.log(cfg.tau_min), np.log(cfg.tau_max), H))
            self.hid_beta_raw = nn.Parameter(torch.tensor([logit(float(np.exp(-(n/cfg.fs)/t))) for t in taus], dtype=torch.float32))
        else:
            self.hid_beta_raw = nn.Parameter(torch.full((H,), logit(0.95**n)))
        self.fc_out = nn.Linear(H, K)
        self.out_beta_raw = nn.Parameter(torch.full((K,), logit(0.95**n)))

    def forward(self, x, record=False):
        s = self.cfg.surr_slope
        bands = self.frontend(x)
        B, C, T = bands.shape
        be = torch.sigmoid(self.enc_beta_raw); thp = F.softplus(self.enc_th_pos_raw); thn = F.softplus(self.enc_th_neg_raw)
        up = x.new_zeros(B, C); un = x.new_zeros(B, C); ZP, ZN = [], []
        for t in range(T):                                    # encoder loop (independent of the hidden layer)
            b = bands[:, :, t]
            up = be*up + b; un = be*un - b
            zp = spike(up - thp, s); zn = spike(un - thn, s)
            up = up - zp*thp; un = un - zn*thn
            ZP.append(zp); ZN.append(zn)
        S_in = torch.cat([torch.stack(ZP, 1), torch.stack(ZN, 1)], 2)      # (B, T, 2C)
        I = self.fc_in(S_in)                                                # one matmul for all t
        bh = torch.sigmoid(self.hid_beta_raw)
        p_drop = self.cfg.spike_dropout if self.training else 0.0
        vh = x.new_zeros(B, self.cfg.hidden); sh_out = torch.zeros_like(vh); SH, SHraw = [], []
        for t in range(T):                                    # recurrent loop
            vh = bh*vh + I[:, t] + self.fc_rec(sh_out)
            sh = spike(vh - 1.0, s); vh = vh - sh
            sh_out = F.dropout(sh, p_drop, training=True) if p_drop > 0 else sh
            SH.append(sh_out); SHraw.append(sh)
        SH = torch.stack(SH, 1)                                             # (B, T, H) (dropped spikes in training)
        SHr = torch.stack(SHraw, 1) if p_drop > 0 else SH
        w0 = self.w0; assert T > w0, f"window ({T} steps) shorter than the warm-up ({w0})"
        Wr = readout_weights(torch.sigmoid(self.out_beta_raw), T, w0)
        logits = (self.fc_out(SH)*Wr.unsqueeze(0)).sum(1)/(T - w0)
        aux = {"enc_rate": S_in.mean(), "hid_rate": SHr.mean(), "enc_rate_ch": S_in.detach().mean((0, 1))}
        if record: aux["S_in"], aux["SH"] = S_in[0].detach().cpu(), SHr[0].detach().cpu()
        return logits, aux

def _core_params(model):
    return sum(p.numel() for n, p in model.named_parameters() if not n.startswith(("swt.", "ht.", "enc_gain")))

def gru_hidden_matched(cfg):
    # Hidden size whose GRU + readout parameter count is closest to the SNN core (everything after the shared front end).
    target = _core_params(WaveSpikeNet(cfg)); C = (cfg.levels + 1 - len(cfg.drop_bands)) if cfg.use_wavelet else 1; K = cfg.n_classes
    return min(range(2, 513), key=lambda h: abs(3*(C*h + h*h + 2*h) + (h + 1)*K - target))

class WaveGRUNet(WaveFront):
    """Non-spiking twin (P0.4b): same front end, GRU of matched parameter count, linear readout of the mean state."""
    def __init__(self, cfg, hidden=None):
        super().__init__(); self.cfg = cfg; self._build_frontend(cfg)
        self.h = hidden or gru_hidden_matched(cfg)
        self.gru = nn.GRU(self.C, self.h, batch_first=True)
        self.fc_out = nn.Linear(self.h, cfg.n_classes)
    def forward(self, x, record=False):
        bands = self.frontend(x)                                            # (B, C, T)
        out, _ = self.gru(bands.transpose(1, 2).contiguous())
        assert out.shape[1] > self.w0
        z = x.new_zeros(())
        return self.fc_out(out[:, self.w0:].mean(1)), {"enc_rate": z, "hid_rate": z, "enc_rate_ch": None}

def build_model(cfg):
    return {"snn": WaveSpikeNet, "gru": WaveGRUNet}[cfg.arch](cfg)

def n_params(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)

m = WaveSpikeNet(CFG).to(DEVICE)
_g = copy.deepcopy(CFG); _g.arch = "gru"; mg = WaveGRUNet(_g)
print("SNN trainable params:", n_params(m), "| core (after front end):", _core_params(m),
      "| wavelet params:", m.swt.h0.numel() if CFG.learn_wavelet else 0, "| HT params:", 2*m.C if m.ht is not None else 0,
      "| warm-up:", m.w0, "steps")
print(f"GRU twin: hidden {mg.h}, trainable params {n_params(mg)}, core {_core_params(mg)}")
with torch.no_grad():
    t0 = time.time(); lg, aux = m(torch.tensor(X[:CFG.batch], device=DEVICE)); dt = time.time()-t0
print(f"forward on {CFG.batch} windows: {dt:.1f}s | logits {tuple(lg.shape)} | enc {aux['enc_rate']:.3f} hid {aux['hid_rate']:.3f}")

## §5 Training and evaluation
**Loss:** class-weighted CE + λ_wav·wavelet loss + λ_rate·max(0, hidden rate − target)². The schedule is cosine, and each epoch draws a class-balanced subsample of `max_windows_per_epoch` windows.

- **Model selection (P0.2, changed from v2):**
  - Early stopping uses the **validation loss**, i.e. class-weighted CE with the training-set class weights. Val AUROC and val F1 are still logged.
  - Two checkpoints are kept: **`best`** (lowest val loss) and **`last`** (the final epoch trained, which is the early-stopping epoch if training stopped early). Both are saved to Drive and both are reported.
- **Augmentation (P1.8, `augment=True`, training only):** applied in this order:
  1. amplitude × U(0.7, 1.3);
  2. circular shift by U{0, T−1};
  3. Gaussian noise at an SNR of U(20, 40) dB relative to the window's power;
  4. a baseline sinusoid of U(0.05, 0.4) Hz, random phase, amplitude U(0, 0.5) × the window's SD.

  Windows are already z-scored, and test windows are not augmented, so the amplitude term works as input-gain jitter.
- **Out-of-fold evaluation (P0.3):** for each (variant, fold seed, training seed), the test predictions of all 5 folds are concatenated. The notebook then reports:
  - pooled window-level AUROC and macro-F1;
  - **subject-level AUROC**: the mean P(AF) of a subject's windows, scored against whether the subject has any AF segment;
  - the per-fold numbers.

In [ ]:
def batches(idx, bs):
    for i in range(0, len(idx), bs): yield idx[i:i+bs]

@torch.no_grad()
def predict_logits(model, Xa, idx, bs=128):
    model.eval(); out = []
    for b in batches(np.asarray(idx), bs):
        lg, _ = model(torch.tensor(Xa[b], device=DEVICE)); out.append(lg.float().cpu().numpy())
    return np.concatenate(out) if out else np.zeros((0, model.cfg.n_classes), np.float32)

def softmax_np(lg):
    e = np.exp(lg - lg.max(1, keepdims=True)); return e/e.sum(1, keepdims=True)

def predict(model, idx, bs=128, Xa=None):
    return softmax_np(predict_logits(model, X if Xa is None else Xa, idx, bs))

def weighted_ce_np(lg, yt, cw):
    # identical to F.cross_entropy(lg, yt, weight=cw) (mean reduction), computed in float64
    lg = lg.astype(np.float64); m = lg.max(1, keepdims=True)
    logp = lg - m - np.log(np.exp(lg - m).sum(1, keepdims=True))
    w = cw[yt]; return float((w*(-logp[np.arange(len(yt)), yt])).sum()/w.sum())

def metrics(y_true, prob):
    K = prob.shape[1]; pred = prob.argmax(1); cm = confusion_matrix(y_true, pred, labels=range(K))
    sens, spec = [], []
    for c in range(K):
        tp = cm[c, c]; fn = cm[c].sum()-tp; fp = cm[:, c].sum()-tp; tn = cm.sum()-tp-fn-fp
        sens.append(tp/max(tp+fn, 1)); spec.append(tn/max(tn+fp, 1))
    try:
        auc = roc_auc_score(y_true, prob[:, 1]) if K == 2 else roc_auc_score(y_true, prob, multi_class="ovr", average="macro", labels=range(K))
    except ValueError: auc = float("nan")
    return {"macro_f1": float(f1_score(y_true, pred, average="macro", labels=range(K), zero_division=0)),
            "acc": float((pred == y_true).mean()), "auroc": float(auc),
            "sens": [float(v) for v in sens], "spec": [float(v) for v in spec], "cm": cm.tolist()}

def _auc(lab, score):
    lab = np.asarray(lab); return float(roc_auc_score(lab, score)) if len(np.unique(lab)) == 2 else float("nan")

def subject_level(p_af, is_af, g):
    """Subject-level AUROC: mean window P(AF) per subject vs 'subject has any AF window'."""
    df = pd.DataFrame({"g": g, "p": p_af, "a": is_af}).groupby("g").agg(p=("p", "mean"), a=("a", "max"))
    return _auc(df.a.values, df.p.values), len(df)

def binary_af_metrics(p_af, is_af, g):
    is_af = np.asarray(is_af).astype(int)
    sa, ns = subject_level(p_af, is_af, g)
    return {"auroc": _auc(is_af, p_af), "macro_f1": float(f1_score(is_af, (p_af >= 0.5).astype(int), average="macro", zero_division=0)),
            "subj_auroc": sa, "n_subjects": ns}

def balanced_subsample(idx, n, rng, ya=None):
    ya = y if ya is None else ya
    if len(idx) <= n: return rng.permutation(idx)
    cls = [idx[ya[idx] == c] for c in np.unique(ya[idx])]; per = n//len(cls)
    return rng.permutation(np.concatenate([rng.choice(c, min(per, len(c)), replace=False) for c in cls]))

def augment_batch(xb, rng, fs):
    # P1.8 training-time augmentation (see §5 text). xb: (B, T) float32, already z-scored.
    B, T = xb.shape
    x = xb*rng.uniform(0.7, 1.3, (B, 1))
    sh = rng.integers(0, T, B); x = x[np.arange(B)[:, None], (np.arange(T)[None, :] - sh[:, None]) % T]
    p = x.var(1, keepdims=True) + 1e-12
    x = x + rng.standard_normal((B, T))*np.sqrt(p/10**(rng.uniform(20, 40, (B, 1))/10))
    t = np.arange(T)[None, :]/fs
    x = x + rng.uniform(0, 0.5, (B, 1))*np.sqrt(p)*np.sin(2*np.pi*rng.uniform(0.05, 0.4, (B, 1))*t + rng.uniform(0, 2*np.pi, (B, 1)))
    return x.astype(np.float32)

def augment_plus_batch(xb, rng, fs):
    # v3.1 training-time augmentation aimed at making the model rely on beat TIMING, not morphology or heart rate:
    # (1) global time-warp by U(0.85, 1.15) (circular, linear interpolation): changes heart rate, keeps relative irregularity;
    # (2) slow multiplicative amplitude modulation (0.05-0.5 Hz, depth up to 0.3). Output is re-standardised per window.
    B, T = xb.shape
    r = rng.uniform(0.85, 1.15, (B, 1)); pos = (np.arange(T)[None, :]*r + rng.uniform(0, T, (B, 1))) % T
    i0 = np.floor(pos).astype(int); fr = pos - i0; i1 = (i0 + 1) % T
    x = xb[np.arange(B)[:, None], i0]*(1 - fr) + xb[np.arange(B)[:, None], i1]*fr
    t = np.arange(T)[None, :]/fs
    x = x*(1 + rng.uniform(0, 0.3, (B, 1))*np.sin(2*np.pi*rng.uniform(0.05, 0.5, (B, 1))*t + rng.uniform(0, 2*np.pi, (B, 1))))
    x = (x - x.mean(1, keepdims=True))/(x.std(1, keepdims=True) + 1e-6)
    return x.astype(np.float32)

def train(cfg, tr_idx, va_idx, verbose=True, Xa=None, ya=None):
    """Returns {"best": model at the lowest val loss (or best val F1 if select_on="val_f1"), "last": model after the last
    epoch run, "hist", "best_epoch", "best_state", "last_state"}."""
    Xa = X if Xa is None else Xa; ya = y if ya is None else ya
    tr_idx, va_idx = np.asarray(tr_idx), np.asarray(va_idx)
    set_seed(cfg.seed); rng = np.random.default_rng(cfg.seed); arng = np.random.default_rng(cfg.seed + 7919)
    model = build_model(cfg).to(DEVICE)
    has_wav = model.swt is not None
    wav = [model.swt.h0] if (has_wav and cfg.learn_wavelet) else []
    rest = [p for n, p in model.named_parameters() if n != "swt.h0"]
    groups_ = [{"params": rest, "lr": cfg.lr}] + ([{"params": wav, "lr": cfg.lr_wavelet, "weight_decay": 0.0}] if wav else [])
    opt = torch.optim.AdamW(groups_, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg.epochs)
    cnt = np.bincount(ya[tr_idx], minlength=cfg.n_classes).astype(float)
    cw_np = cnt.sum()/(cfg.n_classes*np.maximum(cnt, 1))
    cw = torch.tensor(cw_np, dtype=torch.float32, device=DEVICE)
    best, best_state, best_ep, hist, bad = -np.inf, None, -1, [], 0
    for ep in range(cfg.epochs):
        model.train(); t0 = time.time(); L, ER, HR = [], [], []
        for b in batches(balanced_subsample(tr_idx, cfg.max_windows_per_epoch, rng, ya), cfg.batch):
            xb = Xa[b]
            if cfg.augment: xb = augment_batch(xb, arng, cfg.fs)
            if cfg.augment_plus: xb = augment_plus_batch(xb, arng, cfg.fs)
            xb = torch.tensor(xb, device=DEVICE); yb = torch.tensor(ya[b], device=DEVICE)
            lg, aux = model(xb)
            loss = F.cross_entropy(lg, yb, weight=cw)
            if has_wav: loss = loss + cfg.lambda_wav*wavelet_loss(model.swt.h0)
            if cfg.arch == "snn": loss = loss + cfg.lambda_rate*F.relu(aux["hid_rate"] - cfg.target_rate)**2
            opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
            L.append(loss.item()); ER.append(float(aux["enc_rate"].detach())); HR.append(float(aux["hid_rate"].detach()))
        sched.step()
        lg_va = predict_logits(model, Xa, va_idx); mv = metrics(ya[va_idx], softmax_np(lg_va))
        vl = weighted_ce_np(lg_va, ya[va_idx], cw_np)
        h = {"epoch": ep, "loss": float(np.mean(L)), "val_loss": vl, "val_f1": mv["macro_f1"], "val_auroc": mv["auroc"],
             "enc_rate": float(np.mean(ER)), "hid_rate": float(np.mean(HR)),
             "wav_loss": float(wavelet_loss(model.swt.h0).item()) if has_wav else 0.0, "sec": time.time()-t0}
        hist.append(h)
        score = -vl if cfg.select_on == "val_loss" else mv["macro_f1"]
        if score > best: best, best_state, best_ep, bad = score, copy.deepcopy(model.state_dict()), ep, 0
        else: bad += 1
        if verbose:
            print(f"ep {ep:2d} loss {h['loss']:.3f} | val loss {vl:.3f} F1 {h['val_f1']:.3f} AUROC {h['val_auroc']:.3f} | "
                  f"enc {h['enc_rate']:.3f} hid {h['hid_rate']:.3f} | wav {h['wav_loss']:.1e} | {h['sec']:.0f}s" + (" *" if best_ep == ep else ""))
            if cfg.arch == "snn" and h["hid_rate"] < 1e-3: print("  ⚠ hidden layer nearly silent")
        if bad >= cfg.patience:
            if verbose: print(f"early stop (no val improvement for {cfg.patience} epochs); best epoch {best_ep}")
            break
    last_state = copy.deepcopy(model.state_dict()); model.eval()
    best_model = build_model(cfg).to(DEVICE); best_model.load_state_dict(best_state); best_model.eval()
    return {"best": best_model, "last": model, "hist": hist, "best_epoch": best_ep,
            "best_state": best_state, "last_state": last_state}

def report(mt, title="TEST", names=None):
    names = names or CLASS_NAMES
    print(f"{title}: macro-F1 {mt['macro_f1']:.3f} | acc {mt['acc']:.3f} | AUROC {mt['auroc']:.3f}")
    for c, n in enumerate(names): print(f"   {n:8s} sens {mt['sens'][c]:.3f}  spec {mt['spec'][c]:.3f}")
    print("   confusion (rows = true):", mt["cm"])

In [ ]:
# P0.4a baseline features: pulse-interval irregularity (also used for the P1.9 label-noise check)
FEAT_NAMES = ["cv", "rmssd_over_mean", "frac_dI_gt_15pct"]

def pulse_intervals(x, fs):
    # Assumption: a 0.5–8 Hz zero-phase band-pass before find_peaks (baseline wander otherwise breaks the peak picking).
    # Peaks at least 0.27 s apart (<= 222 bpm), prominence >= 0.3 SD of the filtered window.
    xf = sosfiltfilt(butter(2, [0.5, 8.0], btype="band", fs=fs, output="sos"), np.asarray(x, np.float64))
    pk, _ = find_peaks(xf, distance=max(1, int(0.27*fs)), prominence=0.3*xf.std())
    return np.diff(pk)/fs

def irregularity_features(x, fs):
    ii = pulse_intervals(x, fs)
    if len(ii) < 4: return [np.nan]*3
    m = ii.mean(); d = np.diff(ii)
    return [ii.std()/m, np.sqrt(np.mean(d**2))/m, float(np.mean(np.abs(d) > 0.15*m))]

def feature_matrix(Xa, fs, cache=None):
    if cache and os.path.exists(cache): return np.load(cache)["F"]
    Fm = np.array([irregularity_features(x, fs) for x in Xa], np.float64)
    if cache: np.savez(cache, F=Fm)
    return Fm

class IrregLR:
    """Logistic regression on the 3 interval features (NaN -> training median, standardised, balanced class weights)."""
    def __init__(self, K): self.K = K
    def fit(self, Fm, yv):
        self.med = np.nan_to_num(np.nanmedian(Fm, 0), nan=0.0)
        self.clf = make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced", max_iter=2000))
        self.clf.fit(self._imp(Fm), yv); return self
    def _imp(self, Fm): return np.where(np.isnan(Fm), self.med, Fm)
    def predict_proba(self, Fm):
        P = np.zeros((len(Fm), self.K)); P[:, self.clf.classes_] = self.clf.predict_proba(self._imp(Fm)); return P

t0 = time.time()
FEATS = feature_matrix(X, CFG.fs, f"{DATA_ROOT}/processed/feats_{DATASET}_{DATA_FP[:10]}.npz")
FEATS_E = feature_matrix(XE, CFG.fs, f"{DATA_ROOT}/processed/feats_mimic_perform_af_ext.npz") if XE is not None else None
print(f"irregularity features: {FEATS.shape} ({time.time()-t0:.0f}s) | windows with < 4 intervals: {np.isnan(FEATS[:, 0]).mean():.1%}")

# P1.9 label-noise rule. Heuristic thresholds, not validated: "regular" = CV < 0.06 and < 10 % of successive interval
# changes > 15 %; "irregular" = CV > 0.15 and > 40 % of successive changes > 15 %.
REG_RULE   = {"cv_max": 0.06, "frac_max": 0.10}
IRREG_RULE = {"cv_min": 0.15, "frac_min": 0.40}
def looks_regular(Fm):   return (Fm[:, 0] < REG_RULE["cv_max"]) & (Fm[:, 2] < REG_RULE["frac_max"])
def looks_irregular(Fm): return (Fm[:, 0] > IRREG_RULE["cv_min"]) & (Fm[:, 2] > IRREG_RULE["frac_min"])
SR_IDX = CLASS_NAMES.index("SR") if "SR" in CLASS_NAMES else (0 if AF_IDX is not None else None)
def label_disagrees(idx):
    """True where the feature-based rhythm strongly disagrees with the label (AF but regular, SR/non-AF but irregular)."""
    idx = np.asarray(idx); Fm = FEATS[idx]                # NaN features compare False: never regular or irregular
    if AF_IDX is None: return np.zeros(len(idx), bool)
    with np.errstate(invalid="ignore"):
        return ((y[idx] == AF_IDX) & looks_regular(Fm)) | ((y[idx] == SR_IDX) & looks_irregular(Fm))

In [ ]:
# Experiment definitions shared by §6 and §9: variants, one run = (variant, fold seed, training seed, fold)
VARIANTS = {
    "full":          {},                                            # v3 defaults (ht_nonneg=True)
    "fixed_wavelet": {"learn_wavelet": False},
    "shared_filter": {"per_level": False},
    "no_ht":         {"use_ht": False},
    # P0.4 baselines
    "irreg_lr":      {"_kind": "lr"},                               # (a) interval irregularity + logistic regression
    "gru_twin":      {"arch": "gru"},                               # (b) same front end + GRU of matched size
    "snn_raw":       {"use_wavelet": False, "use_ht": False},       # (c) SNN without wavelet
    # P1 fixes, one flag each
    "ht_despawn":    {"ht_nonneg": False},                          # P1.6 the unconstrained v2 HT
    "lean":          {"drop_bands": [0, 1], "snn_decim": 2},        # P1.7
    "aug":           {"augment": True},                             # P1.8
    "spkdrop":       {"spike_dropout": 0.15},                       # P1.8
    "h32":           {"hidden": 32},                                # P1.8
    "reg_all":       {"augment": True, "spike_dropout": 0.15, "hidden": 32},
    "full_clean":    {"_clean": True},                              # P1.9 train/val cleaned, test untouched
    # v3.1 accuracy candidates (all new, defaults unchanged). Screened on fold seed 0 / train seed 0 only.
    "tau_het":       {"tau_het": True},                             # hidden leaks spread over 0.2-4 s (can remember ~1-2 beats)
    "augplus":       {"augment": True, "augment_plus": True},       # time-warp + amplitude modulation on top of aug
    "slowtrain":     {"epochs": 40, "patience": 40, "lr": 1e-3},    # no early stop; cosine to zero over 40 epochs
    "wavlr":         {"lr_wavelet": 5e-3, "lambda_wav": 0.1},       # let the wavelet filters move 10x more
    "raw_tau":       {"use_wavelet": False, "use_ht": False, "tau_het": True},
    "tau_aug":       {"tau_het": True, "augment": True, "augment_plus": True},
    "tau_aug_slow":  {"tau_het": True, "augment": True, "augment_plus": True, "epochs": 40, "patience": 40, "lr": 1e-3},
    "lean_tau_aug":  {"drop_bands": [0, 1], "snn_decim": 2, "tau_het": True, "augment": True, "augment_plus": True},
    # stacked on the best dev-screen variant (tau_aug_slow)
    "raw_tau_aug_slow":  {"use_wavelet": False, "use_ht": False, "tau_het": True, "augment": True, "augment_plus": True, "epochs": 40, "patience": 40, "lr": 1e-3},
    "tas_spkdrop":   {"tau_het": True, "augment": True, "augment_plus": True, "epochs": 40, "patience": 40, "lr": 1e-3, "spike_dropout": 0.15},
    "tas_wavlr":     {"tau_het": True, "augment": True, "augment_plus": True, "epochs": 40, "patience": 40, "lr": 1e-3, "lr_wavelet": 5e-3, "lambda_wav": 0.1},
    "tas_60":        {"tau_het": True, "augment": True, "augment_plus": True, "epochs": 60, "patience": 60, "lr": 1e-3},
    "tas_h128":      {"tau_het": True, "augment": True, "augment_plus": True, "epochs": 40, "patience": 40, "lr": 1e-3, "hidden": 128},
    "lean_tas":      {"drop_bands": [0, 1], "snn_decim": 2, "tau_het": True, "augment": True, "augment_plus": True, "epochs": 40, "patience": 40, "lr": 1e-3},
}
EXPORTABLE = [v for v, s in VARIANTS.items() if s.get("arch", "snn") == "snn" and s.get("_kind") != "lr"]

RES_DIR = f"{DATA_ROOT}/results/{DATASET}_v3_{DATA_FP[:8]}"; CKPT_DIR = f"{RES_DIR}/ckpt"
os.makedirs(CKPT_DIR, exist_ok=True)
print("results:", RES_DIR)

def variant_config(v, seed, base=None):
    c = copy.deepcopy(base or CFG); c.seed = seed
    for k, val in VARIANTS[v].items():
        if not k.startswith("_"): setattr(c, k, copy.deepcopy(val))
    return c

def run_path(v, cs, ts, k): return f"{RES_DIR}/{v}_cv{cs}_ts{ts}_fold{k}.json"

def _probs_entry(P, PE, te):
    r = metrics(y[te], P); r["prob"] = np.round(P, 5).tolist()
    if AF_IDX is not None: r["subj_auroc"] = subject_level(P[:, AF_IDX], (y[te] == AF_IDX).astype(int), groups[te])[0]
    if PE is not None and AF_IDX is not None:
        r["ext"] = binary_af_metrics(PE[:, AF_IDX], yE, gE); r["ext_p_af"] = np.round(PE[:, AF_IDX], 5).tolist()
    return r

def save_run(v, cs, ts, k, te, entries, extra):
    rec = {"variant": v, "cv_seed": cs, "train_seed": ts, "fold": k, "data_fp": DATA_FP, "te_idx": np.asarray(te).tolist(),
           "external_overlap": EXTERNAL_OVERLAP_STATUS, **entries, **extra}
    p = run_path(v, cs, ts, k); json.dump(rec, open(p + ".tmp", "w")); os.replace(p + ".tmp", p)   # atomic on Drive
    return rec

def run_one(v, cs, ts, k, verbose=False):
    tr, va, te = FOLDS_BY_CV[cs][k]; spec = VARIANTS[v]; t0 = time.time()
    if spec.get("_clean"):
        n0 = len(tr) + len(va); tr, va = tr[~label_disagrees(tr)], va[~label_disagrees(va)]
        if verbose: print(f"  cleaned train+val: {n0} -> {len(tr)+len(va)} windows (test untouched)")
    if spec.get("_kind") == "lr":
        trva = np.concatenate([tr, va]); lr_ = IrregLR(CFG.n_classes).fit(FEATS[trva], y[trva])
        P = lr_.predict_proba(FEATS[te]); PE = lr_.predict_proba(FEATS_E) if XE is not None else None
        e = _probs_entry(P, PE, te); entries = {"best": e, "last": e}; extra = {"epochs_run": 0, "best_epoch": -1, "hist": []}
    else:
        c = variant_config(v, ts); out = train(c, tr, va, verbose=verbose)
        tag = f"{CKPT_DIR}/{v}_cv{cs}_ts{ts}_fold{k}"
        torch.save(out["best_state"], tag + "_best.pt"); torch.save(out["last_state"], tag + "_last.pt")
        entries = {}
        for name in ("best", "last"):
            mdl = out[name]
            PE = predict(mdl, np.arange(len(XE)), Xa=XE) if XE is not None else None
            entries[name] = _probs_entry(predict(mdl, te), PE, te)
        extra = {"epochs_run": len(out["hist"]), "best_epoch": out["best_epoch"],
                 "hist": [{k_: (round(v_, 5) if isinstance(v_, float) else v_) for k_, v_ in h.items()} for h in out["hist"]]}
    extra["minutes"] = (time.time() - t0)/60
    return save_run(v, cs, ts, k, te, entries, extra)

In [ ]:
# §6 Main run: fold 0, trained TWICE with identical settings (P0.1 determinism check)
CS0, TS0, K0 = CV_SEEDS[0], TRAIN_SEEDS[0], 0
tr, va, te = FOLDS_BY_CV[CS0][K0]
c0 = variant_config("full", TS0)
FORCE_DETERMINISM_CHECK = False      # True: train fold 0 twice again even if an earlier session already verified it
DET_MARK = f"{RES_DIR}/determinism_check.json"
_ck = f"{CKPT_DIR}/full_cv{CS0}_ts{TS0}_fold{K0}"
_resume = (os.path.exists(DET_MARK) and os.path.exists(run_path("full", CS0, TS0, K0)) and os.path.exists(_ck + "_best.pt")
           and not FORCE_DETERMINISM_CHECK)
RUNS = []
if _resume:                          # earlier session: verified, load the saved fold-0 run instead of training twice more
    _rec = json.load(open(run_path("full", CS0, TS0, K0))); out = {"hist": _rec["hist"], "best_epoch": _rec["best_epoch"]}
    for name in ("best", "last"):
        out[name + "_state"] = torch.load(f"{_ck}_{name}.pt", map_location=DEVICE)
        out[name] = build_model(c0).to(DEVICE); out[name].load_state_dict(out[name + "_state"]); out[name].eval()
    RUNS.append({"out": out, "P_best": predict(out["best"], te), "P_last": predict(out["last"], te), "sec": _rec["minutes"]*60})
    print("P0.1 determinism was verified in an earlier session:", json.load(open(DET_MARK)), "| loaded the saved fold-0 run")
for rep in range(0 if _resume else 2):
    print(f"--- training run {rep+1}/2 (full, fold seed {CS0}, train seed {TS0}, fold {K0}) ---")
    try:
        t0 = time.time(); out = train(c0, tr, va, verbose=(rep == 0))
    except RuntimeError as e:
        if "deterministic" in str(e).lower():
            print("P0.1 NONDETERMINISTIC OP (torch refused to run it in deterministic mode):\n   ", str(e).splitlines()[0])
        raise
    RUNS.append({"out": out, "P_best": predict(out["best"], te), "P_last": predict(out["last"], te), "sec": time.time() - t0})

if not _resume:
    same_pred = all(np.array_equal(RUNS[0][k], RUNS[1][k]) for k in ("P_best", "P_last"))
    same_hist = [h["val_loss"] for h in RUNS[0]["out"]["hist"]] == [h["val_loss"] for h in RUNS[1]["out"]["hist"]]
    same_w = all(torch.equal(RUNS[0]["out"]["last_state"][k], RUNS[1]["out"]["last_state"][k]) for k in RUNS[0]["out"]["last_state"])
    m_a, m_b = metrics(y[te], RUNS[0]["P_best"]), metrics(y[te], RUNS[1]["P_best"])
    print(f"P0.1 determinism: test AUROC {m_a['auroc']:.6f} vs {m_b['auroc']:.6f} | identical predictions: {same_pred} | "
          f"identical val-loss curves: {same_hist} | identical weights: {same_w}")
    if not (same_pred and same_hist and same_w):
        d = next((i for i, (a, b) in enumerate(zip(RUNS[0]["out"]["hist"], RUNS[1]["out"]["hist"])) if a["val_loss"] != b["val_loss"]), None)
        raise AssertionError(f"training is not deterministic (first differing epoch: {d}). No op raised in deterministic mode, so "
                             "the difference comes from outside torch's checks (e.g. CUBLAS_WORKSPACE_CONFIG set after CUDA "
                             "started: restart the session).")
    json.dump({"test_auroc_run1": m_a["auroc"], "test_auroc_run2": m_b["auroc"], "identical": True, "torch": torch.__version__,
               "device": str(DEVICE), "date": time.strftime("%Y-%m-%d %H:%M")}, open(DET_MARK, "w"))

out = RUNS[0]["out"]; model, hist = out["best"], out["hist"]
print(f"best epoch {out['best_epoch']} (lowest val loss) of {len(hist)} run")
report(metrics(y[te], RUNS[0]["P_best"]), "TEST fold 0 [best-val-loss checkpoint]")
report(metrics(y[te], RUNS[0]["P_last"]), "TEST fold 0 [last-epoch checkpoint]")
fig, ax = plt.subplots(1, 2, figsize=(12, 3))
ax[0].plot([h["loss"] for h in hist], label="train loss"); ax[0].plot([h["val_loss"] for h in hist], label="val loss")
ax[0].axvline(out["best_epoch"], color="k", ls=":"); ax[0].legend(); ax[0].set_xlabel("epoch")
ax[1].plot([h["val_auroc"] for h in hist], label="val AUROC"); ax[1].plot([h["val_f1"] for h in hist], label="val F1")
ax[1].axvline(out["best_epoch"], color="k", ls=":"); ax[1].legend(); ax[1].set_xlabel("epoch"); plt.tight_layout(); plt.show()

# This run is (full, CS0, TS0, fold 0) of the ablation: save it, so §9 doesn't train it again.
if not os.path.exists(run_path("full", CS0, TS0, K0)):
    tag = f"{CKPT_DIR}/full_cv{CS0}_ts{TS0}_fold{K0}"
    torch.save(out["best_state"], tag + "_best.pt"); torch.save(out["last_state"], tag + "_last.pt")
    ents = {}
    for name in ("best", "last"):
        PE = predict(out[name], np.arange(len(XE)), Xa=XE) if XE is not None else None
        ents[name] = _probs_entry(RUNS[0]["P_" + name], PE, te)
    save_run("full", CS0, TS0, K0, te, ents, {"epochs_run": len(hist), "best_epoch": out["best_epoch"], "minutes": RUNS[0]["sec"]/60,
             "hist": [{k_: (round(v_, 5) if isinstance(v_, float) else v_) for k_, v_ in h.items()} for h in hist]})
    if XE is not None:
        print(f"external MIMIC PERform AF [{EXTERNAL_OVERLAP_STATUS}]:", {k: round(v, 3) for k, v in ents["best"]["ext"].items()})

## §7 What did the wavelets learn?
- **Left:** the learned low-pass taps for each level (dashed = db4 init).
- **Right:** the equivalent band responses, with the heart-rate range (0.7–3 Hz) shaded.
- **Below:** the learned hard thresholds b⁺/b⁻ per band and the **HT gain at x → 0**, σ(−αb⁻)+σ(−αb⁺). Above 1 means the layer amplifies small coefficients; with `ht_nonneg=True` it is ≤ 1 by construction.

In [ ]:
def band_responses(H0, per_level, levels, fs, n=4096):
    sw = LearnableSWT(levels, np.asarray(H0)[0], learnable=False, per_level=per_level)
    sw.h0 = torch.tensor(np.asarray(H0), dtype=torch.float32)
    imp = torch.zeros(1, n); imp[0, 0] = 1.0
    with torch.no_grad(): r = sw(imp)[0].numpy()
    return np.fft.rfftfreq(n, 1/fs), np.abs(np.fft.rfft(r, axis=1))

H_init = np.tile(init_lowpass(CFG.wavelet_init, CFG.taps), (CFG.levels if CFG.per_level else 1, 1))
H_learn = model.swt.h0.detach().cpu().numpy()
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
for j, h in enumerate(H_learn): ax[0].plot(h, "o-", ms=3, label=f"level {j+1}" if CFG.per_level else "shared")
ax[0].plot(H_init[0], "k--", lw=1, label="db4 init"); ax[0].legend(fontsize=7); ax[0].set_title("low-pass taps per level")
for H, ls in [(H_init, ":"), (H_learn, "-")]:
    f, R = band_responses(H, CFG.per_level, CFG.levels, CFG.fs)
    for j in range(R.shape[0]): ax[1].plot(f, R[j], ls, color=f"C{j}", label=(f"d{j+1}" if j < CFG.levels else f"a{CFG.levels}") if ls == "-" else None)
ax[1].axvspan(0.7, 3.0, color="k", alpha=.07); ax[1].set_xscale("log"); ax[1].set_xlim(0.1, CFG.fs/2)
ax[1].set_xlabel("Hz"); ax[1].set_title("band responses (dotted = init)"); ax[1].legend(fontsize=7, ncol=2)
plt.tight_layout(); plt.show()
print("wavelet loss after training: %.2e" % wavelet_loss(model.swt.h0).item())
if model.ht is not None:
    with torch.no_grad(): _bp, _bn = [t.cpu().numpy() for t in model.ht.thresholds()]; _g0 = model.ht.gain_at_zero().cpu().numpy()
    print(f"HT ({'nonneg, b = softplus(raw)' if model.ht.nonneg else 'unconstrained DeSpaWN'}) bands {model.band_names}")
    print("   b+          :", np.round(_bp, 3))
    print("   b-          :", np.round(_bn, 3))
    print("   gain at x→0 :", np.round(_g0, 3), "(> 1 = amplifies small coefficients)")
print("encoder θ+ :", np.round(F.softplus(model.enc_th_pos_raw).detach().cpu().numpy(), 3))
print("encoder θ- :", np.round(F.softplus(model.enc_th_neg_raw).detach().cpu().numpy(), 3))

In [ ]:
# §8 Spike raster for one test window + per-band encoder rates over 128 test windows (dead-band check, P1.7)
with torch.no_grad(): _, aux = model(torch.tensor(X[te[:1]], device=DEVICE), record=True)
enc, hid = aux["S_in"].numpy(), aux["SH"].numpy(); tt = np.arange(enc.shape[0])*model.decim/CFG.fs
fig, ax = plt.subplots(3, 1, figsize=(12, 6), sharex=True)
ax[0].plot(np.arange(T_WIN)/CFG.fs, X[te[0]], lw=.8); ax[0].set_title(f"true: {CLASS_NAMES[y[te[0]]]}")
lbl = [f"+{n}" for n in model.band_names] + [f"−{n}" for n in model.band_names]
for k in range(enc.shape[1]): ax[1].plot(tt[enc[:, k] > 0], np.full((enc[:, k] > 0).sum(), k), "|", ms=4)
ax[1].set_yticks(range(len(lbl))); ax[1].set_yticklabels(lbl, fontsize=6)
for k in range(hid.shape[1]): ax[2].plot(tt[hid[:, k] > 0], np.full((hid[:, k] > 0).sum(), k), "|", ms=2, color="k")
ax[2].axvline(model.w0*model.decim/CFG.fs, color="r", ls="--", lw=1, label="readout starts (filter warm-up)"); ax[2].legend(fontsize=7)
ax[2].set_xlabel("s"); plt.tight_layout(); plt.show()
with torch.no_grad(): _, aux = model(torch.tensor(X[te[:128]], device=DEVICE))
rates = aux["enc_rate_ch"].cpu().numpy()
print("per-band encoder rate (128 test windows):", {k: round(float(v), 4) for k, v in zip(lbl, rates)})
print("dead bands (rate of both + and − < 0.005):",
      [n for i, n in enumerate(model.band_names) if max(rates[i], rates[i + model.C]) < 0.005] or "none")

## §9 Repeated-CV ablation (resumable)
- **Resuming:** one JSON per (variant, fold seed, training seed, fold) is written to `results/<dataset>_v3_<data hash>/` on Drive, together with both checkpoints. Re-running the cell skips completed runs.
- **Run order:** repeat (fold seed × training seed) → fold → variant. Whenever a session ends, every variant therefore has the same set of completed repeats, which keeps the paired comparison valid.
- **Session limit:** no new run is started after `MAX_SESSION_H` hours. Run the notebook again in a new session to continue.
- **Time estimate (printed before launching):**
  - Seconds per epoch come from the **first epoch of the §6 run**.
  - The expected epochs per run are the number §6 needed before early stopping.
  - Each variant's speed relative to `full` comes from timing a few training batches.

**What the analysis reports (P0.3, P0.5):**
- Each (fold seed, training seed) **repeat** gives one pooled out-of-fold AUROC (window-level and subject-level) per variant.
- The table shows mean ± SD over repeats, plus the per-fold mean.
- The **paired difference** (variant − full) is computed per repeat. An improvement is only called when the difference has the same sign in **every** repeat and at least 6 repeats exist. The Wilcoxon p-value is shown for information only: with 6 pairs the smallest possible two-sided p is 0.031.
- **External (MIMIC PERform AF):** each fold model is evaluated on all external windows. The fold average is one value per repeat. It is reported only when the D6 overlap check was verified.

In [ ]:
# §9a Variants to run + time estimate (printed BEFORE the ablation is launched)
RUN_VARIANTS = ["full", "irreg_lr", "gru_twin", "snn_raw", "fixed_wavelet", "ht_despawn", "lean", "aug", "spkdrop", "h32"]
# also available: "shared_filter", "no_ht", "reg_all"
if LABEL_CLEAN_TRAIN: RUN_VARIANTS.append("full_clean")
REPEATS = [(cs, ts) for cs in CV_SEEDS for ts in TRAIN_SEEDS]
# v3.1: optional overrides for screening runs (unset = exactly the plan above). WAVESPIKE_VARIANTS="a,b"  WAVESPIKE_REPEATS="0:0,3:1" (cv:ts)
if os.environ.get("WAVESPIKE_VARIANTS"): RUN_VARIANTS = os.environ["WAVESPIKE_VARIANTS"].split(",")
if os.environ.get("WAVESPIKE_REPEATS"): REPEATS = [tuple(int(a) for a in r.split(":")) for r in os.environ["WAVESPIKE_REPEATS"].split(",")]
for _cs, _ts in REPEATS:
    if _cs not in FOLDS_BY_CV: FOLDS_BY_CV[_cs] = make_folds(STRAT, groups, CFG.n_folds, seed=_cs)

def time_train_batches(cfg, n=3):
    set_seed(0); mdl = build_model(cfg).to(DEVICE); mdl.train()
    opt = torch.optim.AdamW(mdl.parameters(), lr=1e-4); b = FOLDS[0][0][:cfg.batch]
    xb, yb = torch.tensor(X[b], device=DEVICE), torch.tensor(y[b], device=DEVICE)
    def step():
        lg, _ = mdl(xb); loss = F.cross_entropy(lg, yb); opt.zero_grad(); loss.backward(); opt.step()
    step()
    if DEVICE.type == "cuda": torch.cuda.synchronize()
    t0 = time.time()
    for _ in range(n): step()
    if DEVICE.type == "cuda": torch.cuda.synchronize()
    return (time.time() - t0)/n

ep_sec_full = hist[0]["sec"]                       # first epoch of the §6 run (train + validation)
epochs_est = len(hist)                             # epochs §6 needed (early stopping)
n_ext = len(XE) if XE is not None else 0
_t0 = time.time(); predict(model, FOLDS[0][2][:256]); _pw = (time.time() - _t0)/min(256, len(FOLDS[0][2]))
eval_sec_full = 2*(len(FOLDS[0][2]) + n_ext)*_pw    # best + last checkpoint on test + external
tb = {}
for v in RUN_VARIANTS:
    if VARIANTS[v].get("_kind") == "lr": continue
    tb[v] = time_train_batches(variant_config(v, 0))
if "full" not in tb: tb["full"] = time_train_batches(variant_config("full", 0))      # reference for the relative timings
est = {}
for v in RUN_VARIANTS:
    est[v] = 0.05 if VARIANTS[v].get("_kind") == "lr" else (epochs_est*ep_sec_full + eval_sec_full) * tb[v]/tb["full"] / 60
todo = {v: sum(not os.path.exists(run_path(v, cs, ts, k)) for cs, ts in REPEATS for k in range(CFG.n_folds)) for v in RUN_VARIANTS}
total_h = sum(est[v]*todo[v] for v in RUN_VARIANTS)/60
per_repeat_h = sum(est[v] for v in RUN_VARIANTS)*CFG.n_folds/60
left_h = MAX_SESSION_H - (time.time() - NOTEBOOK_T0)/3600
print(f"TIME ESTIMATE from the first §6 epoch ({ep_sec_full:.0f} s/epoch for 'full') × {epochs_est} epochs/run (as in §6):")
for v in RUN_VARIANTS:
    print(f"  {v:14s} ≈ {est[v]:6.1f} min/run" + (f"  (batch time × {tb[v]/tb['full']:.2f} vs full)" if v in tb else "") +
          f" | runs to do {todo[v]:3d}")
print(f"  total remaining ≈ {total_h:.1f} h | one repeat (all variants × {CFG.n_folds} folds) ≈ {per_repeat_h:.1f} h | "
      f"{len(REPEATS)} repeats planned")
print(f"  this session has ≈ {left_h:.1f} h left before MAX_SESSION_H -> ≈ {max(0, left_h)/max(per_repeat_h, 1e-9):.1f} repeats; "
      f"≈ {max(1, math.ceil(total_h/MAX_SESSION_H))} sessions in total")
if total_h > left_h:
    print("  ⚠ The full ablation doesn't fit in one session. It is resumable: re-run the notebook in new sessions. To cut "
          "the time instead, shorten RUN_VARIANTS or TRAIN_SEEDS, but no improvement claim is made with fewer than 6 repeats.")

In [ ]:
# §9b Run the ablation (resumable; stops launching new runs after MAX_SESSION_H hours)
# PARALLEL (env WAVESPIKE_PARALLEL=1, off by default): several processes run this cell at once on the same RES_DIR.
# Each run is claimed with an exclusive lock file first; a lock whose process is gone (same host) is taken over.
# Runs are still visited in repeat -> fold -> variant order, so the finished set stays close to complete repeats.
PARALLEL = os.environ.get("WAVESPIKE_PARALLEL") == "1"
LOCK_DIR = f"{RES_DIR}/locks"
import socket
_HOST = socket.gethostname()

def _claim(p):
    if not PARALLEL: return True
    os.makedirs(LOCK_DIR, exist_ok=True); lk = f"{LOCK_DIR}/{os.path.basename(p)}.lock"
    for _ in range(2):
        try:
            fd = os.open(lk, os.O_CREAT | os.O_EXCL | os.O_WRONLY); os.write(fd, f"{_HOST} {os.getpid()}".encode()); os.close(fd)
            if os.path.exists(p): os.remove(lk); return False         # finished by another worker in the meantime
            return True
        except FileExistsError:
            try: host, pid = open(lk).read().split()
            except (OSError, ValueError): return False                # being written right now
            if host != _HOST: return False
            try: os.kill(int(pid), 0); return False                   # owner still alive
            except ProcessLookupError: os.remove(lk)                  # stale: owner died; retry once
            except PermissionError: return False
    return False

def _release(p):
    if PARALLEL:
        try: os.remove(f"{LOCK_DIR}/{os.path.basename(p)}.lock")
        except FileNotFoundError: pass

if PARALLEL: print(f"PARALLEL worker pid {os.getpid()} on {_HOST}: claiming runs via {LOCK_DIR}")
_stop = False
for cs, ts in REPEATS:
    for k in range(CFG.n_folds):
        for v in RUN_VARIANTS:
            p = run_path(v, cs, ts, k)
            if os.path.exists(p): continue
            if (time.time() - NOTEBOOK_T0)/3600 > MAX_SESSION_H:
                print(f"⏸ session budget of {MAX_SESSION_H} h reached; run the notebook again to resume."); _stop = True; break
            if not _claim(p): continue
            try: r = run_one(v, cs, ts, k)
            finally: _release(p)
            print(f"{v:14s} cv{cs} ts{ts} fold {k}: AUROC best {r['best']['auroc']:.3f} / last {r['last']['auroc']:.3f} | "
                  f"epochs {r['epochs_run']} (best {r['best_epoch']}) | {r['minutes']:.1f} min", flush=True)
        if _stop: break
    if _stop: break

In [ ]:
# §9c Results: pooled out-of-fold metrics per repeat -> mean ± SD over repeats, paired differences vs "full"
def load_runs(res_dir=None):
    R = []
    for p in glob.glob(f"{res_dir or RES_DIR}/*_cv*_ts*_fold*.json"):
        r = json.load(open(p))
        if r.get("data_fp") == DATA_FP: R.append(r)
    return R

def pooled_table(R, ckpt):
    by = {}
    for r in R: by.setdefault((r["variant"], r["cv_seed"], r["train_seed"]), []).append(r)
    rows = []
    for (v, cs, ts), rr in sorted(by.items()):
        if len({r["fold"] for r in rr}) < CFG.n_folds: continue           # incomplete repeat
        rr = sorted(rr, key=lambda r: r["fold"])
        idx = np.concatenate([r["te_idx"] for r in rr]); P = np.concatenate([np.array(r[ckpt]["prob"]) for r in rr]); P = P/P.sum(1, keepdims=True)   # undo rounding drift
        assert np.array_equal(np.sort(idx), np.arange(len(y))), "pooled OOF predictions must cover every window exactly once"
        mm = metrics(y[idx], P)
        row = {"variant": v, "cv_seed": cs, "train_seed": ts, "oof_auroc": mm["auroc"], "oof_f1": mm["macro_f1"],
               "fold_auroc_mean": float(np.nanmean([r[ckpt]["auroc"] for r in rr])),
               "fold_auroc_sd": float(np.nanstd([r[ckpt]["auroc"] for r in rr], ddof=1)),
               "epochs": float(np.mean([r["epochs_run"] for r in rr]))}
        if AF_IDX is not None:
            row["oof_subj_auroc"] = subject_level(P[:, AF_IDX], (y[idx] == AF_IDX).astype(int), groups[idx])[0]
        if all("ext" in r[ckpt] for r in rr):
            for k_ in ("auroc", "subj_auroc", "macro_f1"):
                row["ext_" + k_] = float(np.nanmean([r[ckpt]["ext"][k_] for r in rr]))
        rows.append(row)
    return pd.DataFrame(rows)

def _ms(s):
    s = s.dropna()
    return "—" if len(s) == 0 else (f"{s.mean():.3f} ± {s.std(ddof=1):.3f}" if len(s) > 1 else f"{s.mean():.3f}")

def summarize(df, metric_cols, delta_cols=("oof_auroc", "oof_subj_auroc"), ref="full"):
    out = []
    for v in [v for v in VARIANTS if v in set(df.variant)]:
        d = df[df.variant == v]; row = {"variant": v, "repeats": len(d)}
        for c in metric_cols: row[c] = _ms(d[c]) if c in d else "—"
        if v != ref and ref in set(df.variant):
            mg = d.merge(df[df.variant == ref], on=["cv_seed", "train_seed"], suffixes=("", "_ref"))
            for c in delta_cols:
                if c not in mg: continue
                diff = (mg[c] - mg[c + "_ref"]).dropna().values
                row[f"Δ{c} vs {ref}"] = _ms(pd.Series(diff))
                if c == delta_cols[0]:
                    n, pos, neg = len(diff), int((diff > 0).sum()), int((diff < 0).sum())
                    p = wilcoxon(diff).pvalue if n >= 6 and np.any(diff != 0) else float("nan")
                    row[f"better/worse ({c})"] = f"{pos}/{neg} of {n}"
                    row["wilcoxon p"] = f"{p:.3f}" if p == p else "—"
                    row["verdict"] = ("too few repeats (<6)" if n < 6 else "BETTER in every repeat" if pos == n else
                                      "WORSE in every repeat" if neg == n else "no consistent difference")
        out.append(row)
    return pd.DataFrame(out).fillna("—")

R_ALL = load_runs()
int_cols = ["oof_auroc", "oof_f1", "oof_subj_auroc", "fold_auroc_mean", "epochs"]
ext_cols = ["ext_auroc", "ext_subj_auroc", "ext_macro_f1"]
EXT_DELTAS = ("ext_auroc", "ext_subj_auroc")
TABLES = {}
for ckpt in ("best", "last"):
    df = pooled_table(R_ALL, ckpt)
    if df.empty: print(f"[{ckpt}] no complete repeat yet"); continue
    TABLES[ckpt] = summarize(df, [c for c in int_cols if c in df.columns])
    df.to_csv(f"{RES_DIR}/per_repeat_{ckpt}.csv", index=False)
    print(f"\n=== {DATASET} | checkpoint: {ckpt} | internal {CFG.n_folds}-fold out-of-fold | mean ± SD over repeats ===")
    with pd.option_context("display.width", 250, "display.max_columns", None):
        print(TABLES[ckpt].to_string(index=False))
    if XE is not None and "ext_auroc" in df:
        T_ext = summarize(df, ext_cols, EXT_DELTAS)
        if EXTERNAL_OVERLAP_STATUS == "verified":
            TABLES[ckpt + " | external MIMIC PERform AF (fold-model average)"] = T_ext
            print(f"\n=== external MIMIC PERform AF (overlap check verified) | checkpoint: {ckpt} ===")
        else:
            print(f"\n⚠ external MIMIC PERform AF [{EXTERNAL_OVERLAP_STATUS}] — NOT part of the results table:")
        with pd.option_context("display.width", 250, "display.max_columns", None):
            print(T_ext.to_string(index=False))

if TABLES:
    with open(f"{RES_DIR}/results_table.md", "w") as f:
        f.write(f"# WaveSpike-PPG v3 results: {DATASET}\n\nData fingerprint `{DATA_FP[:8]}`, {len(np.unique(groups))} subjects, "
                f"{len(y)} windows. Repeats = fold seeds {CV_SEEDS} × training seeds {TRAIN_SEEDS}. External overlap check: "
                f"{EXTERNAL_OVERLAP_STATUS}.\nImprovement is claimed only if the paired Δ has the same sign in every repeat (≥ 6).\n\n")
        for ckpt, T_ in TABLES.items():
            try: body = T_.to_markdown(index=False)
            except ImportError: body = "```\n" + T_.to_string(index=False) + "\n```"
            f.write(f"## Checkpoint: {ckpt}\n\n" + body + "\n\n")
    for ckpt, T_ in TABLES.items(): T_.to_csv(f"{RES_DIR}/results_table_{re.sub(r'[^A-Za-z0-9]+', '_', ckpt).strip('_')}.csv", index=False)
    print("\nwritten:", f"{RES_DIR}/results_table.md")

In [ ]:
# P1.9 Label-noise check: interval-irregularity features per class (labels from chart notes up to 15 min from the waveform)
fig, ax = plt.subplots(1, 3, figsize=(15, 3.2))
for j, name in enumerate(FEAT_NAMES):
    for c, cn in enumerate(CLASS_NAMES):
        v = FEATS[y == c, j]; v = v[np.isfinite(v)]
        ax[j].hist(np.clip(v, 0, np.nanpercentile(FEATS[:, j], 99)), bins=60, alpha=.5, density=True, label=f"{cn} (n={len(v)})")
    ax[j].set_title(name); ax[j].legend(fontsize=7)
ax[0].axvline(REG_RULE["cv_max"], color="g", ls=":"); ax[0].axvline(IRREG_RULE["cv_min"], color="r", ls=":")
ax[2].axvline(REG_RULE["frac_max"], color="g", ls=":"); ax[2].axvline(IRREG_RULE["frac_min"], color="r", ls=":")
plt.tight_layout(); plt.show()
with np.errstate(invalid="ignore"):
    _reg, _irr = looks_regular(FEATS), looks_irregular(FEATS)
_ok = np.isfinite(FEATS[:, 0])
print(f"rule: regular = {REG_RULE}, irregular = {IRREG_RULE} (heuristic thresholds); windows without enough peaks: {(~_ok).mean():.1%}")
for c, cn in enumerate(CLASS_NAMES):
    m = (y == c) & _ok
    print(f"  {cn:6s}: {m.sum():6d} windows with features | look regular {_reg[m].mean():6.1%} | look irregular {_irr[m].mean():6.1%}")
if AF_IDX is not None:
    _dis = label_disagrees(np.arange(len(y)))
    print(f"AF-labelled windows that look regular: {_reg[(y == AF_IDX) & _ok].mean():.1%} | "
          f"{CLASS_NAMES[SR_IDX]}-labelled windows that look irregular: {_irr[(y == SR_IDX) & _ok].mean():.1%} | "
          f"strong disagreements: {_dis.sum()} windows ({_dis.mean():.1%}) in {len(set(groups[_dis]))} subjects")
    print("LABEL_CLEAN_TRAIN =", LABEL_CLEAN_TRAIN, "-> when True, variant 'full_clean' drops these windows from TRAIN and VAL "
          "only. Test folds stay uncleaned, so the classifier is never scored on data filtered by its own rule. Note: the "
          "'irreg_lr' baseline uses the same features.")

## §10 Export for Raspberry Pi (NumPy only) and equivalence check
- **Main file:** the §6 model is exported to `wavespike_rpi.npz`.
- **Equivalence check:** every exportable configuration (`ht_nonneg` on/off, `drop_bands` + `snn_decim`, no wavelet, `hidden=32`, spike dropout, fixed/shared filters, no HT) is checked against `rpi_infer.py` on 50 test windows. It must give **100 % prediction agreement**, and the cell raises otherwise.
- **Latency:** it is measured on the Colab CPU, so it is only a relative comparison. The `lean` vs `full` latency is the P1.7 latency change; measure again on the Pi.
- **Not exported:** the GRU twin and the logistic-regression baseline.

In [ ]:
RPI_SRC = r'''
# rpi_infer.py: NumPy-only inference for WaveSpike-PPG v3 (exported from the Colab notebook).
# Usage:  python3 rpi_infer.py wavespike_rpi.npz        (self-test on a synthetic pulse)
# Replace read_window_from_sensor() with your sensor driver. The window length must match training (cfg win_sec at 50 Hz).
# Supports the v3 flags ht_nonneg, drop_bands, snn_decim and use_wavelet; files exported by v2 still load (v2 defaults).
import sys, time, json, numpy as np

def _sigmoid(x): return 1.0/(1.0+np.exp(-x))
def _softplus(x): return np.log1p(np.exp(x))

class WaveSpikeNP:
    def __init__(self, path):
        d = np.load(path); f = np.float32
        self.cfg = c = json.loads(str(d["config"]))
        if c.get("arch", "snn") != "snn": raise ValueError("only the spiking model can be exported")
        self.levels = int(c["levels"]); self.use_wavelet = bool(c.get("use_wavelet", True))
        self.H0 = np.atleast_2d(d["h0"]).astype(f); self.L = self.H0.shape[1]
        self.H1 = np.stack([np.array([(-1.0)**k*h[self.L-1-k] for k in range(self.L)], dtype=f) for h in self.H0])
        n_all = self.levels + 1 if self.use_wavelet else 1
        drop = set(int(b) for b in c.get("drop_bands", []))
        self.keep = [k for k in range(n_all) if k not in drop]          # 0 = d1 ... levels-1 = d<levels>, levels = approx
        self.decim = int(c.get("snn_decim", 1))
        self.use_ht = bool(c["use_ht"]) and self.use_wavelet; self.alpha = f(c["ht_alpha"])
        if self.use_ht and c.get("ht_nonneg", False):                  # b = softplus(raw) >= 0: shrink-only threshold
            self.bp, self.bn = _softplus(d["ht_b_pos_raw"].astype(f)).astype(f), _softplus(d["ht_b_neg_raw"].astype(f)).astype(f)
        elif self.use_ht:
            self.bp, self.bn = d["ht_b_pos"].astype(f), d["ht_b_neg"].astype(f)
        self.gain = d["enc_gain"].astype(f); self.be = _sigmoid(d["enc_beta_raw"]).astype(f)
        self.thp = _softplus(d["enc_th_pos_raw"]).astype(f); self.thn = _softplus(d["enc_th_neg_raw"]).astype(f)
        self.W_in, self.b_in = d["fc_in.weight"].astype(f), d["fc_in.bias"].astype(f)
        self.W_rec = d["fc_rec.weight"].astype(f); self.bh = _sigmoid(d["hid_beta_raw"]).astype(f)
        self.W_out, self.b_out = d["fc_out.weight"].astype(f), d["fc_out.bias"].astype(f)
        self.bo = _sigmoid(d["out_beta_raw"]).astype(f)
        w0 = sum((2**j)*(self.L-1) for j in range(self.levels)) if self.use_wavelet else 0
        self.warmup = -(-w0 // self.decim)                              # in SNN steps (after decimation)
        self.classes = c["class_names"]

    def _lp(self, j): return (self.H0[j], self.H1[j]) if self.H0.shape[0] > 1 else (self.H0[0], self.H1[0])

    def bands(self, x):
        # causal stationary wavelet transform: y[t] = sum_k h[k] a[t-d*k] / sqrt(2), per-level filters.
        # Only the kept detail bands are computed (the low-pass chain is always needed).
        a = x.astype(np.float32)
        if not self.use_wavelet: return a[None, :]
        T = len(a); out = {}; s = np.float32(1/np.sqrt(2))
        for j in range(self.levels):
            h0, h1 = self._lp(j); d = 2**j; ap = np.concatenate([np.zeros(d*(self.L-1), np.float32), a])
            lo = np.zeros(T, np.float32); hi = np.zeros(T, np.float32) if j in self.keep else None
            for k in range(self.L):
                seg = ap[d*(self.L-1-k): d*(self.L-1-k)+T]; lo += h0[k]*seg
                if hi is not None: hi += h1[k]*seg
            if hi is not None: out[j] = hi*s
            a = lo*s
        out[self.levels] = a
        return np.stack([out[k] for k in self.keep])

    def encoder_input(self, x):
        # normalise -> SWT (kept bands) -> HT -> gain -> decimation; returns (C, T // decim) float32
        x = np.asarray(x, np.float32); x = (x - x.mean())/(x.std() + 1e-6)
        bands = self.bands(x)
        if self.use_ht:
            bp, bn = self.bp[:, None], self.bn[:, None]
            bands = bands*(_sigmoid(-self.alpha*(bands + bn)) + _sigmoid(self.alpha*(bands - bp)))
        bands = (bands*self.gain[:, None]).astype(np.float32)
        if self.decim > 1: bands = np.ascontiguousarray(bands[:, ::self.decim])   # all kept bands < fs/(2*decim)
        return bands

    def classify_window(self, x, stats=None):
        # x: one window of raw samples at 50 Hz. Returns (class, probs, hidden spike rate, logits).
        # stats: optional dict; if given, spike counts are accumulated in it (benchmarking only, does not change the result).
        bands = self.encoder_input(x)
        C, T = bands.shape; H = self.W_rec.shape[0]; K = self.W_out.shape[0]
        up = np.zeros(C, np.float32); un = np.zeros(C, np.float32)
        vh = np.zeros(H, np.float32); sh = np.zeros(H, np.float32); vo = np.zeros(K, np.float32)
        acc = np.zeros(K, np.float32); nspk = 0.0
        for t in range(T):
            b = bands[:, t]
            up = self.be*up + b; un = self.be*un - b
            zp = (up - self.thp >= 0).astype(np.float32); zn = (un - self.thn >= 0).astype(np.float32)
            up -= zp*self.thp; un -= zn*self.thn
            act = np.nonzero(np.concatenate([zp, zn]))[0]       # event-driven: only active input columns
            vh = self.bh*vh + self.W_in[:, act].sum(1) + self.b_in + self.W_rec @ sh
            sh = (vh - 1.0 >= 0).astype(np.float32); vh -= sh
            if stats is not None:
                stats["steps"] = stats.get("steps", 0) + 1; stats["in_spikes"] = stats.get("in_spikes", 0) + len(act)
                stats["hid_spikes"] = stats.get("hid_spikes", 0) + int(sh.sum())
            vo = self.bo*vo + self.W_out @ sh + self.b_out
            if t >= self.warmup: acc += vo
            nspk += sh.mean()
        logits = acc/(T - self.warmup); p = np.exp(logits - logits.max()); p /= p.sum()
        return self.classes[int(p.argmax())], p, nspk/T, logits

def read_window_from_sensor(n, fs=50):
    # Replace with your sensor driver (e.g. a MAX3010x over I2C); must return n samples at fs Hz.
    t = np.arange(n)/fs; return np.sin(2*np.pi*1.2*t) + 0.3*np.sin(2*np.pi*2.4*t)

if __name__ == "__main__":
    net = WaveSpikeNP(sys.argv[1] if len(sys.argv) > 1 else "wavespike_rpi.npz")
    n = int(net.cfg["fs"]*net.cfg["win_sec"]); x = read_window_from_sensor(n)
    t0 = time.perf_counter(); cls, p, rate, _ = net.classify_window(x); dt = time.perf_counter() - t0
    print(f"prediction {cls} | probs {np.round(p, 3)} | hidden rate {rate:.3f} | {dt*1e3:.1f} ms per {net.cfg['win_sec']} s window")
'''
with open("rpi_infer.py", "w") as f: f.write(RPI_SRC)
import rpi_infer

def export_rpi(model, cfg, path):
    assert cfg.arch == "snn", "only the SNN can be exported"
    sd = {k: v.detach().cpu().numpy() for k, v in model.state_dict().items()}; C = model.C
    exp = {"h0": sd.get("swt.h0", np.zeros((1, cfg.taps), np.float32)),
           "enc_gain": sd["enc_gain"], "enc_beta_raw": sd["enc_beta_raw"],
           "enc_th_pos_raw": sd["enc_th_pos_raw"], "enc_th_neg_raw": sd["enc_th_neg_raw"],
           "fc_in.weight": sd["fc_in.weight"], "fc_in.bias": sd["fc_in.bias"], "fc_rec.weight": sd["fc_rec.weight"],
           "hid_beta_raw": sd["hid_beta_raw"], "fc_out.weight": sd["fc_out.weight"], "fc_out.bias": sd["fc_out.bias"],
           "out_beta_raw": sd["out_beta_raw"],
           "config": json.dumps({**asdict(cfg), "class_names": CLASS_NAMES, "dataset": DATASET})}
    if model.ht is not None and cfg.ht_nonneg: exp["ht_b_pos_raw"], exp["ht_b_neg_raw"] = sd["ht.b_pos_raw"], sd["ht.b_neg_raw"]
    elif model.ht is not None:                 exp["ht_b_pos"], exp["ht_b_neg"] = sd["ht.b_pos"], sd["ht.b_neg"]
    np.savez(path, **exp)

def equivalence_check(model, cfg, idx, path, n_lat=5):
    export_rpi(model, cfg, path); importlib.reload(rpi_infer); net = rpi_infer.WaveSpikeNP(path)
    lg_t = predict_logits(model, X, idx)
    with torch.no_grad():
        xn = torch.tensor(normalize_windows(X[idx[:8]]), device=DEVICE)
        fe_t = model.frontend(xn).cpu().numpy()
    fe_n = np.stack([net.encoder_input(X[i]) for i in idx[:8]])
    assert fe_t.shape == fe_n.shape, (fe_t.shape, fe_n.shape)
    res = [net.classify_window(X[i]) for i in idx]; lg_n = np.stack([r[3] for r in res])
    t0 = time.perf_counter(); [net.classify_window(X[i]) for i in idx[:n_lat]]; ms = (time.perf_counter() - t0)/n_lat*1e3
    return {"agree": float((lg_t.argmax(1) == lg_n.argmax(1)).mean()), "max_dlogit": float(np.abs(lg_t - lg_n).max()),
            "max_dband": float(np.abs(fe_t - fe_n).max()),
            "ms_per_window": ms, "hid_rate": float(np.mean([r[2] for r in res])), "steps": net.cfg["fs"]*net.cfg["win_sec"]//net.decim}

# 1) the §6 model -> wavespike_rpi.npz (the file for the Pi)
r_main = equivalence_check(model, c0, te[:50], "wavespike_rpi.npz")
if USE_DRIVE:
    shutil.copy("wavespike_rpi.npz", f"{DATA_ROOT}/results/wavespike_rpi_{DATASET}.npz"); shutil.copy("rpi_infer.py", f"{DATA_ROOT}/results/rpi_infer.py")
print("saved wavespike_rpi.npz (%.1f kB) + rpi_infer.py | §6 model: max |logit diff| = %.2e | agreement = %.0f%% | %.0f ms/window"
      % (os.path.getsize("wavespike_rpi.npz")/1024, r_main["max_dlogit"], 100*r_main["agree"], r_main["ms_per_window"]))

# 2) every exportable variant (v3 flags): use the ablation checkpoint (fold seed 0, train seed 0, fold 0) if it exists,
#    otherwise a 1-epoch model, so that spikes actually occur in the check
EQUIV_VARIANTS = ["full", "ht_despawn", "lean", "snn_raw", "h32", "spkdrop", "fixed_wavelet", "shared_filter", "no_ht"]
rows = []
for v in EQUIV_VARIANTS:
    c = variant_config(v, TS0); ck = f"{CKPT_DIR}/{v}_cv{CS0}_ts{TS0}_fold{K0}_best.pt"
    if os.path.exists(ck):
        mdl = build_model(c).to(DEVICE); mdl.load_state_dict(torch.load(ck, map_location=DEVICE)); src = "ablation ckpt"
    else:
        cq = copy.deepcopy(c); cq.epochs = 1; cq.max_windows_per_epoch = min(256, CFG.max_windows_per_epoch)
        mdl = train(cq, tr, va[:128], verbose=False)["last"]; src = "1-epoch model"
    r = equivalence_check(mdl, c, te[:50], f"_equiv_{v}.npz"); os.remove(f"_equiv_{v}.npz")
    rows.append({"variant": v, "weights": src, "agreement": f"{100*r['agree']:.0f}%", "max|Δlogit|": f"{r['max_dlogit']:.1e}", "max|Δband|": f"{r['max_dband']:.1e}",
                 "SNN steps": r["steps"], "ms/window": round(r["ms_per_window"], 1), "hidden rate": round(r["hid_rate"], 3)})
EQ = pd.DataFrame(rows); print(EQ.to_string(index=False))
lat = dict(zip(EQ.variant, EQ["ms/window"]))
print(f"P1.7 latency on THIS machine: full {lat['full']:.0f} ms -> lean (drop d1,d2 + decim 2) {lat['lean']:.0f} ms per window "
      f"({100*(lat['lean']/lat['full'] - 1):+.0f}%). Measure again on the Pi.")
assert all(r_["agreement"] == "100%" for r_ in rows) and r_main["agree"] == 1.0, "PyTorch vs NumPy predictions disagree"
assert max(float(r_["max|Δband|"]) for r_ in rows) < 1e-3, "front end (SWT/HT/drop/decim) differs between PyTorch and NumPy"
print("§10 equivalence check passed: 100% prediction agreement for every exported configuration.")

## §11 On the Raspberry Pi
```bash
pip3 install numpy
python3 rpi_infer.py wavespike_rpi.npz     # self-test: prints prediction and latency
```
- **Sensor:** replace `read_window_from_sensor()` with your driver. It must deliver `win_sec × 50` samples at 50 Hz.
- **Running:** classify a sliding buffer, e.g. every 5 s.
- **Measure for the paper:**
  - latency per window
  - CPU %
  - peak memory (`/usr/bin/time -v`)
  - power (USB power meter)
  - hidden-layer spike rate

A finger sensor on the Pi is a different domain from ICU or wrist training data, so validate on recordings from your own setup.

## D7 (on by default): PhysioNet/CinC Challenge 2015 false-alarm sanity task
**This is not AF detection** and it is reported separately.
- **Data:** open access, 250 Hz, 5-min records (some 5.5 min), with the alarm at 5:00. Only the alarm types Tachycardia and Ventricular_Tachycardia are kept, and only from records that have a PLETH channel.
- **Input:** the 30 s of PLETH just before the alarm (270–300 s), resampled 250 → 50 Hz with `resample_poly(1, 5)`.
- **Task:** true vs false alarm.
- **Evaluation:** 5-fold stratified CV, grouped by record, with `full` vs `irreg_lr`. Patient IDs are not in the headers, so records are assumed to come from different patients.
- **Labels:** the alarm type and the true/false label are read from the header comments, and the parsed comments are printed. If they can't be parsed, the cell stops instead of guessing.

Set `RUN_CINC2015 = False` in §2 to skip it.

In [ ]:
# D7 CinC 2015 sanity task (optional)
CINC_BASE = "https://physionet.org/files/challenge-2015/1.0.0/"
CINC_TYPES = {"tachycardia": "Tachycardia", "ventricular_tachycardia": "Ventricular_Tachycardia"}

def _norm(s): return re.sub(r"[^a-z]+", "_", str(s).lower()).strip("_")

def load_cinc2015(cfg):
    import wfdb
    raw = f"{DATA_ROOT}/raw/cinc2015"; cache = f"{DATA_ROOT}/processed/cinc2015_tachy_vt_pleth_{cfg.fs}hz_30s.npz"
    if os.path.exists(cache):
        d = np.load(cache); print("loaded cache", cache); return d["X"], d["y"], d["rec"].astype(str), d["atype"].astype(str)
    if fetch_physionet(["training/RECORDS"], CINC_BASE, raw, None): raise RuntimeError("training/RECORDS not found")
    recs = [l.strip() for l in open(f"{raw}/training/RECORDS") if l.strip()]
    fetch_physionet([f"training/{r}.hea" for r in recs], CINC_BASE, raw, None, desc="CinC headers")
    sel, unparsed = [], []
    for r in recs:
        if not _nonempty(f"{raw}/training/{r}.hea"): continue
        hd = wfdb.rdheader(f"{raw}/training/{r}"); com = [_norm(c) for c in (hd.comments or [])]
        atype = next((CINC_TYPES[c] for c in com if c in CINC_TYPES), None)
        known_type = any(c in {"asystole", "bradycardia", "ventricular_flutter_fib"} | set(CINC_TYPES) for c in com)
        truth = 1 if "true_alarm" in com else 0 if "false_alarm" in com else None
        if truth is None or not known_type: unparsed.append((r, hd.comments)); continue
        if atype and any("pleth" in s.lower() for s in hd.sig_name): sel.append((r, atype, truth, hd))
    print(f"D7 headers: {len(recs)} records, {len(unparsed)} unparsed, {len(sel)} Tachycardia/VT with PLETH. "
          f"Example comments: {wfdb.rdheader(f'{raw}/training/{recs[0]}').comments}")
    if len(unparsed) == len(recs):
        raise StopForUser(f"No alarm type / true-false label found in the header comments (e.g. {unparsed[0]}). Adapt load_cinc2015.")
    files = sorted({f"training/{f}" for _, _, _, hd in sel for f in set(hd.file_name)})
    fetch_physionet(files, CINC_BASE, raw, None, desc="CinC signals")
    Xc, yc, rc, ac = [], [], [], []
    for r, atype, truth, hd in sel:
        rec = wfdb.rdrecord(f"{raw}/training/{r}"); ch = next(i for i, s in enumerate(rec.sig_name) if "pleth" in s.lower())
        fs = int(round(rec.fs)); x = rec.p_signal[int(270*fs):int(300*fs), ch]
        if len(x) < 30*fs: continue
        x, why = clean_segment(x, fs)
        if x is None: continue
        g = math.gcd(fs, cfg.fs); Xc.append(resample_poly(x, cfg.fs//g, fs//g).astype(np.float32)); yc.append(truth); rc.append(r); ac.append(atype)
    Xc, yc, rc, ac = np.stack(Xc), np.array(yc), np.array(rc), np.array(ac)
    np.savez_compressed(cache, X=Xc, y=yc, rec=rc, atype=ac); return Xc, yc, rc, ac

if RUN_CINC2015:
    Xc, yc, rc, ac = load_cinc2015(CFG); Xc = normalize_windows(Xc)
    print(f"D7: {len(yc)} windows | true alarms {int(yc.sum())} / false {int((1-yc).sum())} | by type {Counter(ac)}")
    cfg_c = variant_config("full", 0); cfg_c.n_classes = 2
    folds_c = make_folds(yc, rc, 5, seed=0); Fc = feature_matrix(Xc, CFG.fs)
    P_snn, P_lr, idx_all = [], [], []
    for k, (tr_c, va_c, te_c) in enumerate(folds_c):
        o = train(cfg_c, tr_c, va_c, verbose=False, Xa=Xc, ya=yc)
        P_snn.append(predict(o["best"], te_c, Xa=Xc))
        trva = np.concatenate([tr_c, va_c]); P_lr.append(IrregLR(2).fit(Fc[trva], yc[trva]).predict_proba(Fc[te_c])); idx_all.append(te_c)
    idx_all = np.concatenate(idx_all)
    D7_RESULT = {name: {"oof_auroc": _auc(yc[idx_all], np.concatenate(P)[:, 1]),
                        "oof_f1": metrics(yc[idx_all], np.concatenate(P))["macro_f1"]} for name, P in (("full", P_snn), ("irreg_lr", P_lr))}
    os.makedirs(f"{DATA_ROOT}/results/cinc2015_v3", exist_ok=True)
    json.dump(D7_RESULT, open(f"{DATA_ROOT}/results/cinc2015_v3/oof.json", "w"))
    print("D7 CinC 2015 Tachy/VT true-vs-false alarm (NOT AF detection), pooled OOF, 1 fold seed x 1 training seed:", D7_RESULT)
else:
    print("D7 skipped (RUN_CINC2015 = False)")

## Changes in v3 and assumptions

### Update: open-access data only
This project now uses **only open-access datasets** (`OPEN_ACCESS_ONLY = True`, a new flag in §2).

| Setting | Before | Now |
|---|---|---|
| `DATASET` | `"mimic3_ext_ppg"` (credentialed) | `"mimic_perform_af"` (open, same as v2) |
| `RUN_CINC2015` | `False` | `True` (open; sanity task, not AF detection) |
| External test | MIMIC PERform AF | none (D6 inactive: no second open PPG-AF dataset is confirmed) |
| Non-open datasets | selectable | blocked: choosing `mimic3_ext_ppg`, `deepbeat`, `uconn_*` or `pulsewatch` stops the run and names the access requirement. Set `OPEN_ACCESS_ONLY = False` to allow them again |

The Ext-PPG code is kept unchanged but does not run. No PhysioNet account or Colab secrets are needed.

**Consequence:** training and evaluation are back on 35 subjects. Repeated CV, out-of-fold metrics and the "same sign in every repeat" rule still apply, but small differences between variants are unlikely to be resolvable.

**Candidate second open dataset, not added:** Zenodo 10.5281/zenodo.23597128, "Long-term ECG and wrist PPG recordings with annotated AF episodes". Its licence and access terms are reported inconsistently (one index says an application is required), and zenodo.org was not reachable when this was written. Check the record page before using it.

### Defaults that changed (all others are as in v2)
| Setting | v2 | v3 | Why |
|---|---|---|---|
| `DATASET` | `"mimic_perform_af"` | `"mimic3_ext_ppg"` | D1: 35 subjects is too few to compare methods |
| `Config.ht_nonneg` (new) | (unconstrained HT) | `True`: b = softplus(raw), raw₀ = inv_softplus(0.02) | P1.6. The old HT is `ht_nonneg=False` (variant `ht_despawn`) |
| `Config.select_on` (new) | early stopping on val macro-F1 | `"val_loss"` (class-weighted CE, training-set weights) | P0.2. `"val_f1"` restores v2 |
| `train()` return value | `(model, hist)` | dict with `best`, `last`, `hist`, `best_epoch`, `best_state`, `last_state` | P0.2: both checkpoints |
| Determinism | off | `CUBLAS_WORKSPACE_CONFIG=":4096:8"`, `use_deterministic_algorithms(True)`, cudnn deterministic, no benchmark | P0.1 |
| Data root outside Colab | crashed (Drive mount) | `$WAVESPIKE_DATA_ROOT` or `/content/wavespike_data` | Needed for the local test; no effect in Colab |
| Results folder | `results/<dataset>/<variant>_fold<k>_seed<s>.json` | `results/<dataset>_v3_<data hash>/<variant>_cv<c>_ts<t>_fold<k>.json` (+ `ckpt/`) | New fields; v2 JSONs are not mixed in |
| HT parameter names when `ht_nonneg=True` | `ht.b_pos`, `ht.b_neg` | `ht.b_pos_raw`, `ht.b_neg_raw` | Both are mirrored in `rpi_infer.py`, and v2 `.npz` exports still load (tested) |

**New flags, off by default (v2 behaviour unchanged):** `use_wavelet=True`, `drop_bands=[]`, `snn_decim=1`, `arch="snn"`, `spike_dropout=0.0`, `augment=False`, `EXT_MULTICLASS=False`, `RUN_CINC2015=False`, `LABEL_CLEAN_TRAIN=False`.

**Leak initialisation with decimation:** with `snn_decim=n`, every leak β is initialised to β₀ⁿ, which keeps the time constants in seconds. For n = 1 this is identical to v2.

### Code changes, by item
- **D1** `physionet_creds()` reads `PHYSIONET_USER` / `PHYSIONET_PASS` from `google.colab.userdata`. They are never printed, and wget stderr is scrubbed. The other steps:
  - `physionet_listing()` lists the project, and README* + `metadata.csv` are fetched with `wget -N -c --user … --password …` into `MyDrive/wavespike_data/raw/mimic3_ext_ppg/`.
  - `ext_d1_inspect()` prints the README, the columns and example values.
  - `ext_d1_resolve()` **stops** (`StopForUser`) until `EXT_SCHEMA` (subject, label, path columns) is set by hand.
  - The label codes must match `EXT_LABEL_CODES` **exactly**. The run stops if any value contains a code alongside other text (e.g. "AF, PVC"), or if one path has several labels or subjects.
- **D2** Only AF and SR are kept. `EXT_MULTICLASS=True` keeps SR, AF, STACH and SBRAD. AF is class index 1 in both cases.
- **D3** `ext_select()` works in two stages:
  1. **Subjects:** `N_SUBJECTS=600` are picked with `EXT_SELECT_SEED=0`, stratified by "has any AF segment", half each where available.
  2. **Segments:** up to `MAX_SEG_PER_SUBJECT=40` per subject, round-robin over classes, which balances AF/SR where possible.

  Then:
  - A manifest CSV (`subject, record, label, path, downloaded, status`) is written and reused by later runs.
  - Three headers are checked first (125 Hz, a PPG channel, 25–31 s).
  - Only the selected `.hea` files are fetched, then the signal files those headers name. Files already on Drive are skipped.
  - Downloads go to local disk and are copied to Drive only when complete.
- **D4** `pick_ppg_channel()` picks the channel by name (exact `pleth`/`ppg` first, then a substring match). `clean_segment()` drops a segment for any of:
  - NaN or inf;
  - zero variance;
  - < 25 s of valid data, where samples inside runs of identical values lasting ≥ 1 s count as invalid.

  A 25–30 s segment is edge-padded to 30 s. Resampling is `resample_poly(up=2, down=5)`, the cache is `processed/mimic3_ext_ppg_50hz_30s.npz` (with a selection tag, rebuilt when the selection changes), and the counts are printed.
- **D5** `make_folds(STRAT, groups, …)` builds StratifiedGroupKFold folds by subject, stratified on the subject's AF status. The v2 leakage assertions are kept, plus a new one that the test folds partition the data. For MIMIC PERform AF, STRAT equals the window label, so the folds are identical to v2.
- **D6** `mimic_perform_subject_ids()` searches each MIMIC PERform AF record name and its whole `.hea` text for `p\d{6}` or `subject_id <n>`. The rest of the step:
  - It prints the first header and the IDs found.
  - Matching subjects are removed from the Ext-PPG metadata **before** selection, and the removal is asserted again on the final data.
  - If any record has no ID, the status becomes `UNVERIFIED`, a warning is printed, and the external numbers are shown only in a separate, flagged table, never in the results table.
- **D7** `load_cinc2015()` (off by default) uses the open CinC 2015 data:
  - Alarm type and true/false come from the header comments; the run stops if they can't be parsed.
  - Only Tachycardia / Ventricular_Tachycardia records with a PLETH channel are kept.
  - The input is the 270–300 s window, resampled 250 → 50 Hz.
  - Evaluation is 5-fold OOF AUROC for `full` vs `irreg_lr`, saved to `results/cinc2015_v3/`. It is labelled **not AF detection**.
- **P0.1** §6 trains fold 0 twice and asserts identical predictions, val-loss curves and weights. If torch raises a nondeterministic-op error, the op is printed. The result is stored in `determinism_check.json`; later sessions load the saved fold-0 run instead of training it twice again (`FORCE_DETERMINISM_CHECK=True` redoes the check).
- **P0.2** Early stopping is on the weighted val CE, and val AUROC/F1 are logged. The `best` and `last` checkpoints are saved (`ckpt/*.pt`) and reported everywhere. `last` = the final epoch trained, which is the early-stopping epoch when training stopped early.
- **P0.3** Every run JSON stores the test indices and probabilities. `pooled_table()` concatenates the 5 folds and reports:
  - window AUROC and macro-F1;
  - subject AUROC: mean P(AF) per subject vs "subject has any AF window";
  - the per-fold mean.
- **P0.4** Baselines:
  - `irreg_lr`: `find_peaks` on the 50 Hz signal, then CV, RMSSD/mean and frac(|ΔI| > 0.15·mean), fed to a balanced logistic regression;
  - `gru_twin`: the same wavelet/HT front end plus a GRU sized so its parameter count matches the SNN core (hidden 37: 5,109 vs 5,208 core params);
  - `snn_raw`: no wavelet, no HT.

  All run on the same folds.
- **P0.5** Repeats are `CV_SEEDS=[0,1,2]` × `TRAIN_SEEDS=[0,1]`. The table gives mean ± SD over the 6 repeats and the paired Δ vs `full`, both window- and subject-level. The verdict reads "BETTER/WORSE in every repeat" only when all of at least 6 pairs agree; otherwise "no consistent difference" or "too few repeats".
- **P1.6** `LearnableHT(nonneg=True)`; §7 prints the gain at x → 0. The NumPy mirror applies `softplus(raw)`.
- **P1.7** `drop_bands` / `snn_decim` are implemented in `WaveFront.frontend()`: slicing + cat (deterministic backward), then `[::n]`. The warm-up is ⌈w₀/n⌉ (217 → 109). `rpi_infer.py` mirrors this exactly and skips computing the dropped detail bands. §10 prints the latency of `full` vs `lean`; on the local CPU it fell from 68–76 to 33–46 ms per window (−36 to −52 %), to be re-measured on the Pi.
- **P1.8** `augment_batch()` (training only):
  - amplitude × U(0.7, 1.3);
  - circular shift;
  - Gaussian noise at an SNR of U(20, 40) dB;
  - a baseline sinusoid of U(0.05, 0.4) Hz with amplitude U(0, 0.5) × the window's SD.

  Also `spike_dropout` (inverted dropout on hidden spikes, both in the recurrence and at the readout, training only) and `hidden=32`. Variants: `aug`, `spkdrop`, `h32`, `reg_all`.
- **P1.9** Per-class feature histograms and the regular/irregular fractions. Variant `full_clean` (when `LABEL_CLEAN_TRAIN`) drops disagreeing windows from **train and val only**; test folds are never cleaned.
- **§9** The ablation order is repeat → fold → variant, so an interrupted session leaves the paired comparisons complete. The time estimate uses the first §6 epoch × the §6 epoch count, scaled by per-variant batch timing. No new run starts after `MAX_SESSION_H=3.6` h. JSON writes are atomic.
- **§10** `export_rpi()` and `equivalence_check()` cover 9 exportable configurations. The cell asserts 100 % prediction agreement and a front-end difference below 1e-3.

### Assumptions about the file layouts (none checked against the real files; physionet.org and zenodo.org were blocked from where this was built)
- **MIMIC-III-Ext-PPG**
  - **Checked at runtime** (the run stops on a mismatch):
    - `metadata.csv` and a `README*` are at the project root; the root's HTML index lists them.
    - The label column holds the codes `SR`, `AF`, `STACH`, `SBRAD` as whole values. If not, set `EXT_LABEL_CODES`.
    - `path_col` (after `EXT_PATH_TEMPLATE`, with any `.hea`/`.dat` extension stripped) is the record path relative to the project root.
    - Each segment is its own single-segment WFDB record, 125 Hz, about 30 s long, with a channel named like PLETH/PPG.
  - **Not checked:**
    - A segment's signal files are named in its header and sit in the same folder.
    - The subject value contains exactly one digit group, the MIMIC-III subject ID; otherwise the overlap check reports UNVERIFIED.
- **MIMIC PERform AF:**
  - The Zenodo zips are unchanged from v2.
  - The MIMIC-III subject ID may not be in the record names or headers at all. In that case the notebook says so and keeps the external numbers out of the table.
  - Even when the IDs are found, they are only comparable if MIMIC PERform AF comes from the MIMIC-III *matched* subset, where record IDs equal clinical subject IDs.
  - "non-AF" in MIMIC PERform is not necessarily SR.
- **CinC 2015:**
  - `training/RECORDS` exists.
  - Header comments contain the alarm type and "True alarm"/"False alarm".
  - The alarm is at 300 s.
  - Each record is treated as its own group, since patient IDs are not available.
- **Subject-level label:** a subject is "AF" if any of its windows is AF. Ext-PPG subjects can have both AF and SR segments, so for them the subject AUROC is a coarser measure than the window AUROC.
- **Label-noise thresholds** (regular: CV < 0.06 and frac < 0.10; irregular: CV > 0.15 and frac > 0.40) are heuristics, not validated. The peak detector uses a 0.5–8 Hz zero-phase band-pass, peaks ≥ 0.27 s apart, and prominence ≥ 0.3 SD.

### Testing and limits
- **Tested locally on CPU** (torch 2.15), with tiny epochs, on synthetic data and on a **fake** PhysioNet/Zenodo/CinC layout:
  - every code path: D1 stop and resume, schema/label checks, selection, download/skip, cleaning, cache, D6 verified and unverified, multiclass, D7, all 14 variants (`shared_filter`/`no_ht` through the 1-epoch §10 path, `reg_all` separately), OOF pooling, results table, resume of completed runs, and the §10 check (100 %) for all 9 exportable configurations;
  - the standalone `rpi_infer.py` command line;
  - loading of a v2-format export.
- **Not tested:** GPU determinism (whether every CUDA op has a deterministic kernel, e.g. the cuDNN GRU backward; §6 reports any op that hasn't), real Ext-PPG column names, Drive throughput, and T4 timings.
- **Run time:** the full plan of 10 variants × 6 repeats × 5 folds will probably need several T4 sessions. §9a prints the estimate before starting, and the ablation resumes across sessions.